**Code DesiloFHE**

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import time
from desilofhe import Engine

# ─────────────────────────────────────────────────────────────────
def chebyshev_coeffs_step(degree: int, delta: float = 0.05):
    """
    Step fonksiyonu için Chebyshev katsayılarını hesaplar.
    step(x) = 1 if x > 0 else 0
    """
    from scipy.fftpack import dct

    # Grid points
    n = degree + 1
    x = np.cos(np.pi * (np.arange(n) + 0.5) / n)

    # Target function (step at 0)
    # x in [-1, 1]
    y = np.where(x > 0, 1.0, 0.0)

    # DCT to find coefficients
    c = dct(y, type=2, norm='ortho')
    # Normalize for standard Chebyshev definition
    c[0] /= np.sqrt(2)
    c *= (np.sqrt(2.0 / n))
    return c
def eval_chebyshev_plaintext(coeffs, x):
    """Plaintext Chebyshev değerlendirmesi (Validation için)"""
    T = [np.ones_like(x), x]
    for i in range(2, len(coeffs)):
        T.append(2 * x * T[i-1] - T[i-2])

    res = np.zeros_like(x)
    for i, c in enumerate(coeffs):
        res += c * T[i]
    return res
def prepare_data():
    """Iris veriseti üzerinde Logistic Regression eğit."""
    print("[*] Iris veriseti yükleniyor ve model eğitiliyor...")
    iris = load_iris()
    X, y = iris.data, iris.target
    # Binary sınıflandırma: Setosa (0) = 1, Diğerleri = 0
    y_bin = (y == 0).astype(int)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_bin, test_size=0.2, random_state=42
    )
    clf = LogisticRegression(random_state=42)
    clf.fit(X_train, y_train)
    weights = clf.coef_[0]
    bias = clf.intercept_[0]
    train_acc = clf.score(X_train, y_train)
    test_acc = clf.score(X_test, y_test)
    print(f"[✓] Model Eğitildi. Train acc: {train_acc:.4f}, Test acc: {test_acc:.4f}")
    return X_test, y_test, weights, bias, scaler, clf
def run_bootstrapped_fhe():
    X_test, y_test, weights, bias, scaler, clf = prepare_data()
    print("\n" + "=" * 70)
    print("  Bootstrapping: Gerçek FHE Iris Sınıflandırma")
    print("=" * 70)
    # ─────────────────────────────────────────────────────────────────
    # ADIM 0: FHE Engine + Bootstrap Anahtarları
    # ─────────────────────────────────────────────────────────────────
    print("\n[ADIM 0] FHE Engine (bootstrap destekli) başlatılıyor...")
    t0 = time.time()
    # Bootstrapping-enabled motor (Parametreler kütüphane tarafından sabitlenmiştir)
    engine = Engine(use_bootstrap=True)
    print("  → Secret key üretiliyor...")
    sk = engine.create_secret_key()
    print("  → Public key üretiliyor...")
    pk = engine.create_public_key(sk)
    print("  → Relinearization key üretiliyor...")
    rk = engine.create_relinearization_key(sk)
    print("  → Conjugation key üretiliyor...")
    conj_key = engine.create_conjugation_key(sk)
    print("  → Lossy Bootstrap key üretiliyor (bu biraz sürebilir)...")
    t_bk = time.time()
    lbk = engine.create_lossy_bootstrap_key(sk, stage_count=3)
    print(f"  → Bootstrap key hazır. Süre: {time.time() - t_bk:.2f} sn")
    total_setup = time.time() - t0
    print(f"\n[✓] Tüm anahtarlar hazır. Toplam kurulum süresi: {total_setup:.2f} sn")
    print(f"    Slot sayısı: {engine.slot_count}")
    print(f"    Maksimum level: {engine.max_level}")
    # Chebyshev katsayıları
    step_coeffs = chebyshev_coeffs_step(degree=15, delta=0.05)
    # ─────────────────────────────────────────────────────────────────
    # Birden fazla test örneği üzerinde ardışık FHE inference
    # ─────────────────────────────────────────────────────────────────
    n_samples = min(5, len(X_test))
    print(f"\n{'=' * 70}")
    print(f"  {n_samples} test örneği üzerinde şifreli tahmin")
    print(f"{'=' * 70}")
    correct_count = 0
    level = engine.max_level
    for idx in range(n_samples):
        sample = X_test[idx]
        actual = y_test[idx]
        # Plaintext tahmin (doğrulama için)
        logit_pt = np.dot(sample, weights) + bias
        pred_pt = 1 if logit_pt > 0 else 0
        print(f"\n--- Örnek {idx + 1}/{n_samples} ---")
        print(f"  Özellikler: [{', '.join(f'{v:.3f}' for v in sample)}]")
        print(f"  Gerçek sınıf: {actual}  |  Plaintext tahmin: {pred_pt}")
        # ─── ADIM 1: Şifreleme ───
        t_enc = time.time()
        cts = []
        for f_val in sample:
            arr = np.zeros(engine.slot_count)
            arr[0] = f_val
            ct = engine.encrypt(arr, pk)
            cts.append(ct)
        # ─── ADIM 2: Şifreli wx + b ───
        ct_dot = None
        for i, w in enumerate(weights):
            w_arr = np.zeros(engine.slot_count)
            w_arr[0] = w
            pt_w = engine.encode(w_arr, level)
            ct_wx_i = engine.multiply(cts[i], pt_w)
            if ct_dot is None:
                ct_dot = ct_wx_i
            else:
                ct_dot = engine.add(ct_dot, ct_wx_i)
        # Bias ekleme
        b_arr = np.zeros(engine.slot_count)
        b_arr[0] = bias
        pt_b = engine.encode(b_arr, ct_dot.level)
        ct_logit = engine.add(ct_dot, pt_b)
        print(f"  Level (wx+b sonrası): {ct_logit.level}")
        # ─── ADIM 3: Bootstrap (level yenileme) ───
        # Bootstrapping'i GÖSTERMEK için eşiği yüksek tutuyoruz (25 ve altı)
        if ct_logit.level <= 25:
            print(f"  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: {ct_logit.level})...")
            t_bs = time.time()
            ct_logit_intt = engine.intt(ct_logit)
            ct_logit = engine.lossy_bootstrap(
                ct_logit_intt, rk, conj_key, lbk
            )
            print(f"  [BOOTSTRAP] Yeni level: {ct_logit.level}  (süre: {time.time() - t_bs:.2f} sn)")
        # ─── ADIM 4: Normalizasyon + BLEACH Aktivasyonu ───
        # Logit'i [-1, 1] aralığına normalize et
        scale_factor = 20.0
        s_arr = np.zeros(engine.slot_count)
        s_arr[0] = 1.0 / scale_factor
        pt_s = engine.encode(s_arr, ct_logit.level)
        ct_norm = engine.multiply(ct_logit, pt_s)
        print(f"  Level (normalize sonrası): {ct_norm.level}")
        # Level yeterliliğini kontrol et, gerekirse tekrar bootstrap
        if ct_norm.level < 5:
            print(f"  [BOOTSTRAP] Chebyshev öncesi level düşük ({ct_norm.level}), bootstrap...")
            t_bs2 = time.time()
            ct_norm_intt_bs = engine.intt(ct_norm)
            ct_norm = engine.lossy_bootstrap(
                ct_norm_intt_bs, rk, conj_key, lbk
            )
            print(f"  [BOOTSTRAP] Yeni level: {ct_norm.level}  (süre: {time.time() - t_bs2:.2f} sn)")
        # Chebyshev step fonksiyonu
        ct_norm_intt = engine.intt(ct_norm)
        ct_pred = engine.evaluate_chebyshev_polynomial(
            ct_norm_intt, step_coeffs.tolist(), rk
        )
        # ─── ADIM 5: Şifre çözme ───
        result = engine.decrypt(ct_pred, sk)
        fhe_val = result[0]
        fhe_class = int(np.round(fhe_val))
        fhe_class = max(0, min(1, fhe_class))  # 0 veya 1'e clamp
        t_total = time.time() - t_enc
        match = fhe_class == pred_pt
        if match:
            correct_count += 1
        print(f"  FHE ham değer: {fhe_val:.6f}")
        print(f"  FHE tahmin: {fhe_class} {'✓' if match else '✗'}  (süre: {t_total:.2f} sn)")
    # ─── Özet ───
    print(f"\n{'=' * 70}")
    print(f"  SONUÇ: {correct_count}/{n_samples} doğru tahmin")
    print(f"  Doğruluk: {correct_count / n_samples * 100:.1f}%")
    print(f"{'=' * 70}")
    if correct_count == n_samples:
        print("\n[✓] TÜM ŞİFRELİ TAHMİNLER DOĞRU! Bootstrapping başarılı.")
    else:
        print(f"\n[!] {n_samples - correct_count} tahmin uyuşmadı.")
if __name__ == "__main__":
    run_bootstrapped_fhe()

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import time
from desilofhe import Engine


# ─────────────────────────────────────────────────────────────────
# Standalone Fonksiyonları
# ─────────────────────────────────────────────────────────────────

def chebyshev_coeffs_step(degree: int, delta: float = 0.05):
    from scipy.fftpack import dct
    n = degree + 1
    x = np.cos(np.pi * (np.arange(n) + 0.5) / n)
    y = np.where(x > 0, 1.0, 0.0)
    c = dct(y, type=2, norm='ortho')
    c[0] /= np.sqrt(2)
    c *= (np.sqrt(2.0 / n))
    return c


def eval_chebyshev_plaintext(coeffs, x):
    T = [np.ones_like(x), x]
    for i in range(2, len(coeffs)):
        T.append(2 * x * T[i-1] - T[i-2])
    res = np.zeros_like(x)
    for i, c in enumerate(coeffs):
        res += c * T[i]
    return res


def prepare_data():
    print("[*] Iris veriseti yükleniyor ve model eğitiliyor...")
    iris = load_iris()
    X, y = iris.data, iris.target
    y_bin = (y == 0).astype(int)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_bin, test_size=0.2, random_state=42
    )
    clf = LogisticRegression(random_state=42)
    clf.fit(X_train, y_train)
    weights = clf.coef_[0]
    bias = clf.intercept_[0]
    train_acc = clf.score(X_train, y_train)
    test_acc = clf.score(X_test, y_test)
    print(f"[✓] Model Eğitildi. Train acc: {train_acc:.4f}, Test acc: {test_acc:.4f}")
    return X_test, y_test, weights, bias, scaler, clf


def run_bootstrapped_fhe_batched():
    X_test, y_test, weights, bias, scaler, clf = prepare_data()

    print("\n" + "=" * 70)
    print("  BLEACH + Bootstrapping: Batch Paketleme ile Optimize FHE")
    print("=" * 70)

    # ─────────────────────────────────────────────────────────────────
    # ADIM 0: FHE Engine + Bootstrap Anahtarları
    # ─────────────────────────────────────────────────────────────────
    print("\n[ADIM 0] FHE Engine (bootstrap destekli) başlatılıyor...")
    t0 = time.time()

    engine = Engine(use_bootstrap=True)
    print("  → Secret key üretiliyor...")
    sk = engine.create_secret_key()
    print("  → Public key üretiliyor...")
    pk = engine.create_public_key(sk)
    print("  → Relinearization key üretiliyor...")
    rk = engine.create_relinearization_key(sk)
    print("  → Conjugation key üretiliyor...")
    conj_key = engine.create_conjugation_key(sk)
    print("  → Lossy Bootstrap key üretiliyor (bu biraz sürebilir)...")
    t_bk = time.time()
    lbk = engine.create_lossy_bootstrap_key(sk, stage_count=3)
    print(f"  → Bootstrap key hazır. Süre: {time.time() - t_bk:.2f} sn")

    total_setup = time.time() - t0
    print(f"\n[✓] Tüm anahtarlar hazır. Toplam kurulum süresi: {total_setup:.2f} sn")
    print(f"    Slot sayısı: {engine.slot_count}")
    print(f"    Maksimum level: {engine.max_level}")

    step_coeffs = chebyshev_coeffs_step(degree=15, delta=0.05)

    n_samples = min(5, len(X_test))
    n_features = 4
    # Her örnek stride kadar slot kullanıyor, özellikler arka arkaya
    # Örnek i, özellik j → slot[i * stride + j]
    stride = n_features  # 4 — örnekler arasında boşluk yok, bitişik

    print(f"\n{'=' * 70}")
    print(f"  Paketleme: {n_samples} örnek × {n_features} özellik")
    print(f"  Toplam kullanılan slot: {n_samples * stride} / {engine.slot_count}")
    print(f"  Şifreleme sayısı: {n_features} ct (önceki: {n_samples * n_features} ct)")
    print(f"{'=' * 70}")

    # ─────────────────────────────────────────────────────────────────
    # ADIM 1: Tüm örnekleri 4 ciphertext'e paketle
    # ─────────────────────────────────────────────────────────────────
    # ct_feat[j] → tüm örneklerin j. özelliğini içeriyor
    # ct_feat[j][slot i] = X_test[i][j]
    print("\n[ADIM 1] Batch şifreleme...")
    t_enc = time.time()

    ct_feat = []
    for j in range(n_features):
        arr = np.zeros(engine.slot_count)
        for i in range(n_samples):
            arr[i] = X_test[i][j]   # örnek i'nin j. özelliği → slot i
        ct = engine.encrypt(arr, pk)
        ct_feat.append(ct)

    print(f"  → {n_features} ciphertext ile {n_samples} örnek şifrelendi.")
    print(f"  → Şifreleme süresi: {time.time() - t_enc:.2f} sn")

    # ─────────────────────────────────────────────────────────────────
    # ADIM 2: Şifreli wx + b — tüm örnekler paralel
    # ─────────────────────────────────────────────────────────────────
    print("\n[ADIM 2] Şifreli wx + b hesaplanıyor (tüm örnekler paralel)...")

    level = engine.max_level
    ct_dot = None

    for j, w in enumerate(weights):
        # w[j] tüm slotlara aynı değer → örnek i'nin j. özelliğiyle çarpılıyor
        w_arr = np.full(engine.slot_count, w)
        pt_w = engine.encode(w_arr, level)
        ct_wx_j = engine.multiply(ct_feat[j], pt_w)
        if ct_dot is None:
            ct_dot = ct_wx_j
        else:
            ct_dot = engine.add(ct_dot, ct_wx_j)

    # Bias: tüm slotlara aynı bias
    b_arr = np.full(engine.slot_count, bias)
    pt_b = engine.encode(b_arr, ct_dot.level)
    ct_logit = engine.add(ct_dot, pt_b)

    print(f"  → Level (wx+b sonrası): {ct_logit.level}")

    # ─────────────────────────────────────────────────────────────────
    # ADIM 3: Bootstrap — tek seferde tüm örnekler
    # ─────────────────────────────────────────────────────────────────
    if ct_logit.level <= 25:
        print(f"\n[ADIM 3] Bootstrap (tek seferlik, tüm örnekler)...")
        print(f"  Mevcut level: {ct_logit.level}")
        t_bs = time.time()
        ct_logit_intt = engine.intt(ct_logit)
        ct_logit = engine.lossy_bootstrap(ct_logit_intt, rk, conj_key, lbk)
        print(f"  → Yeni level: {ct_logit.level}  (süre: {time.time() - t_bs:.2f} sn)")

    # ─────────────────────────────────────────────────────────────────
    # ADIM 4: Normalizasyon + aktivasyonu — tüm örnekler paralel
    # ─────────────────────────────────────────────────────────────────
    print(f"\n[ADIM 4] Normalizasyon + Chebyshev step (tüm örnekler paralel)...")

    scale_factor = 20.0
    s_arr = np.full(engine.slot_count, 1.0 / scale_factor)
    pt_s = engine.encode(s_arr, ct_logit.level)
    ct_norm = engine.multiply(ct_logit, pt_s)
    print(f"  → Level (normalize sonrası): {ct_norm.level}")

    if ct_norm.level < 5:
        print(f"  [BOOTSTRAP] Chebyshev öncesi level düşük ({ct_norm.level}), bootstrap...")
        t_bs2 = time.time()
        ct_norm_intt_bs = engine.intt(ct_norm)
        ct_norm = engine.lossy_bootstrap(ct_norm_intt_bs, rk, conj_key, lbk)
        print(f"  → Yeni level: {ct_norm.level}  (süre: {time.time() - t_bs2:.2f} sn)")

    ct_norm_intt = engine.intt(ct_norm)
    ct_pred = engine.evaluate_chebyshev_polynomial(
        ct_norm_intt, step_coeffs.tolist(), rk
    )

    # ─────────────────────────────────────────────────────────────────
    # ADIM 5: Şifre çözme — tüm örnekler tek decrypt
    # ─────────────────────────────────────────────────────────────────
    print(f"\n[ADIM 5] Şifre çözme (tek decrypt, tüm örnekler)...")
    result = engine.decrypt(ct_pred, sk)

    # ─────────────────────────────────────────────────────────────────
    # Sonuçları ayıkla ve raporla
    # ─────────────────────────────────────────────────────────────────
    print(f"\n{'=' * 70}")
    print(f"  Sonuçlar")
    print(f"{'=' * 70}")

    correct_count = 0
    for i in range(n_samples):
        fhe_val = result[i]   # örnek i → slot i
        fhe_class = int(np.round(fhe_val))
        fhe_class = max(0, min(1, fhe_class))

        # Plaintext doğrulama
        logit_pt = np.dot(X_test[i], weights) + bias
        pred_pt = 1 if logit_pt > 0 else 0
        actual = y_test[i]
        match = fhe_class == pred_pt

        if match:
            correct_count += 1

        print(f"\n  Örnek {i + 1}/{n_samples}")
        print(f"    Özellikler: [{', '.join(f'{v:.3f}' for v in X_test[i])}]")
        print(f"    Gerçek sınıf: {actual}  |  Plaintext tahmin: {pred_pt}")
        print(f"    FHE ham değer: {fhe_val:.6f}")
        print(f"    FHE tahmin: {fhe_class} {'✓' if match else '✗'}")

    print(f"\n{'=' * 70}")
    print(f"  SONUÇ: {correct_count}/{n_samples} doğru tahmin")
    print(f"  Doğruluk: {correct_count / n_samples * 100:.1f}%")
    print(f"{'=' * 70}")

    if correct_count == n_samples:
        print("\n[✓] TÜM ŞİFRELİ TAHMİNLER DOĞRU! SIMD bootstrapping başarılı.")
    else:
        print(f"\n[!] {n_samples - correct_count} tahmin uyuşmadı.")


if __name__ == "__main__":
    run_bootstrapped_fhe_batched()

30 features code

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import time
from desilofhe import Engine


# ─────────────────────────────────────────────────────────────────
# Standalone Fonksiyonları
# ─────────────────────────────────────────────────────────────────

def chebyshev_coeffs_step(degree: int, delta: float = 0.05):
    from scipy.fftpack import dct
    n = degree + 1
    x = np.cos(np.pi * (np.arange(n) + 0.5) / n)
    y = np.where(x > 0, 1.0, 0.0)
    c = dct(y, type=2, norm='ortho')
    c[0] /= np.sqrt(2)
    c *= (np.sqrt(2.0 / n))
    return c


def eval_chebyshev_plaintext(coeffs, x):
    T = [np.ones_like(x), x]
    for i in range(2, len(coeffs)):
        T.append(2 * x * T[i-1] - T[i-2])
    res = np.zeros_like(x)
    for i, c in enumerate(coeffs):
        res += c * T[i]
    return res


def prepare_data():
    print("[*] Iris veriseti yükleniyor ve model eğitiliyor...")
    iris = load_iris()
    X, y = iris.data, iris.target
    y_bin = (y == 0).astype(int)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_bin, test_size=0.2, random_state=42
    )
    clf = LogisticRegression(random_state=42)
    clf.fit(X_train, y_train)
    weights = clf.coef_[0]
    bias = clf.intercept_[0]
    train_acc = clf.score(X_train, y_train)
    test_acc = clf.score(X_test, y_test)
    print(f"[✓] Model Eğitildi. Train acc: {train_acc:.4f}, Test acc: {test_acc:.4f}")
    return X_test, y_test, weights, bias, scaler, clf


def run_bootstrapped_fhe_batched():
    X_test, y_test, weights, bias, scaler, clf = prepare_data()

    print("\n" + "=" * 70)
    print("  BLEACH + Bootstrapping: Batch Paketleme ile Optimize FHE")
    print("=" * 70)

    # ─────────────────────────────────────────────────────────────────
    # ADIM 0: FHE Engine + Bootstrap Anahtarları
    # ─────────────────────────────────────────────────────────────────
    print("\n[ADIM 0] FHE Engine (bootstrap destekli) başlatılıyor...")
    t0 = time.time()

    engine = Engine(use_bootstrap=True)
    print("  → Secret key üretiliyor...")
    sk = engine.create_secret_key()
    print("  → Public key üretiliyor...")
    pk = engine.create_public_key(sk)
    print("  → Relinearization key üretiliyor...")
    rk = engine.create_relinearization_key(sk)
    print("  → Conjugation key üretiliyor...")
    conj_key = engine.create_conjugation_key(sk)
    print("  → Lossy Bootstrap key üretiliyor (bu biraz sürebilir)...")
    t_bk = time.time()
    lbk = engine.create_lossy_bootstrap_key(sk, stage_count=3)
    print(f"  → Bootstrap key hazır. Süre: {time.time() - t_bk:.2f} sn")

    total_setup = time.time() - t0
    print(f"\n[✓] Tüm anahtarlar hazır. Toplam kurulum süresi: {total_setup:.2f} sn")
    print(f"    Slot sayısı: {engine.slot_count}")
    print(f"    Maksimum level: {engine.max_level}")

    step_coeffs = chebyshev_coeffs_step(degree=15, delta=0.05)

    n_samples = min(30, len(X_test))
    n_features = 4
    # Her örnek stride kadar slot kullanıyor, özellikler arka arkaya
    # Örnek i, özellik j → slot[i * stride + j]
    stride = n_features  # 4 — örnekler arasında boşluk yok, bitişik

    print(f"\n{'=' * 70}")
    print(f"  Paketleme: {n_samples} örnek × {n_features} özellik")
    print(f"  Toplam kullanılan slot: {n_samples * stride} / {engine.slot_count}")
    print(f"  Şifreleme sayısı: {n_features} ct (önceki: {n_samples * n_features} ct)")
    print(f"{'=' * 70}")

    # ─────────────────────────────────────────────────────────────────
    # ADIM 1: Tüm örnekleri 4 ciphertext'e paketle
    # ─────────────────────────────────────────────────────────────────
    # ct_feat[j] → tüm örneklerin j. özelliğini içeriyor
    # ct_feat[j][slot i] = X_test[i][j]
    print("\n[ADIM 1] Batch şifreleme...")
    t_enc = time.time()

    ct_feat = []
    for j in range(n_features):
        arr = np.zeros(engine.slot_count)
        for i in range(n_samples):
            arr[i] = X_test[i][j]   # örnek i'nin j. özelliği → slot i
        ct = engine.encrypt(arr, pk)
        ct_feat.append(ct)

    print(f"  → {n_features} ciphertext ile {n_samples} örnek şifrelendi.")
    print(f"  → Şifreleme süresi: {time.time() - t_enc:.2f} sn")

    # ─────────────────────────────────────────────────────────────────
    # ADIM 2: Şifreli wx + b — tüm örnekler paralel
    # ─────────────────────────────────────────────────────────────────
    print("\n[ADIM 2] Şifreli wx + b hesaplanıyor (tüm örnekler paralel)...")

    level = engine.max_level
    ct_dot = None

    for j, w in enumerate(weights):
        # w[j] tüm slotlara aynı değer → örnek i'nin j. özelliğiyle çarpılıyor
        w_arr = np.full(engine.slot_count, w)
        pt_w = engine.encode(w_arr, level)
        ct_wx_j = engine.multiply(ct_feat[j], pt_w)
        if ct_dot is None:
            ct_dot = ct_wx_j
        else:
            ct_dot = engine.add(ct_dot, ct_wx_j)

    # Bias: tüm slotlara aynı bias
    b_arr = np.full(engine.slot_count, bias)
    pt_b = engine.encode(b_arr, ct_dot.level)
    ct_logit = engine.add(ct_dot, pt_b)

    print(f"  → Level (wx+b sonrası): {ct_logit.level}")

    # ─────────────────────────────────────────────────────────────────
    # ADIM 3: Bootstrap — tek seferde tüm örnekler
    # ─────────────────────────────────────────────────────────────────
    if ct_logit.level <= 25:
        print(f"\n[ADIM 3] Bootstrap (tek seferlik, tüm örnekler)...")
        print(f"  Mevcut level: {ct_logit.level}")
        t_bs = time.time()
        ct_logit_intt = engine.intt(ct_logit)
        ct_logit = engine.lossy_bootstrap(ct_logit_intt, rk, conj_key, lbk)
        print(f"  → Yeni level: {ct_logit.level}  (süre: {time.time() - t_bs:.2f} sn)")

    # ─────────────────────────────────────────────────────────────────
    # ADIM 4: Normalizasyon + aktivasyonu — tüm örnekler paralel
    # ─────────────────────────────────────────────────────────────────
    print(f"\n[ADIM 4] Normalizasyon + Chebyshev step (tüm örnekler paralel)...")

    scale_factor = 20.0
    s_arr = np.full(engine.slot_count, 1.0 / scale_factor)
    pt_s = engine.encode(s_arr, ct_logit.level)
    ct_norm = engine.multiply(ct_logit, pt_s)
    print(f"  → Level (normalize sonrası): {ct_norm.level}")

    if ct_norm.level < 5:
        print(f"  [BOOTSTRAP] Chebyshev öncesi level düşük ({ct_norm.level}), bootstrap...")
        t_bs2 = time.time()
        ct_norm_intt_bs = engine.intt(ct_norm)
        ct_norm = engine.lossy_bootstrap(ct_norm_intt_bs, rk, conj_key, lbk)
        print(f"  → Yeni level: {ct_norm.level}  (süre: {time.time() - t_bs2:.2f} sn)")

    ct_norm_intt = engine.intt(ct_norm)
    ct_pred = engine.evaluate_chebyshev_polynomial(
        ct_norm_intt, step_coeffs.tolist(), rk
    )

    # ─────────────────────────────────────────────────────────────────
    # ADIM 5: Şifre çözme — tüm örnekler tek decrypt
    # ─────────────────────────────────────────────────────────────────
    print(f"\n[ADIM 5] Şifre çözme (tek decrypt, tüm örnekler)...")
    result = engine.decrypt(ct_pred, sk)

    # ─────────────────────────────────────────────────────────────────
    # Sonuçları ayıkla ve raporla
    # ─────────────────────────────────────────────────────────────────
    print(f"\n{'=' * 70}")
    print(f"  Sonuçlar")
    print(f"{'=' * 70}")

    correct_count = 0
    for i in range(n_samples):
        fhe_val = result[i]   # örnek i → slot i
        fhe_class = int(np.round(fhe_val))
        fhe_class = max(0, min(1, fhe_class))

        # Plaintext doğrulama
        logit_pt = np.dot(X_test[i], weights) + bias
        pred_pt = 1 if logit_pt > 0 else 0
        actual = y_test[i]
        match = fhe_class == pred_pt

        if match:
            correct_count += 1

        print(f"\n  Örnek {i + 1}/{n_samples}")
        print(f"    Özellikler: [{', '.join(f'{v:.3f}' for v in X_test[i])}]")
        print(f"    Gerçek sınıf: {actual}  |  Plaintext tahmin: {pred_pt}")
        print(f"    FHE ham değer: {fhe_val:.6f}")
        print(f"    FHE tahmin: {fhe_class} {'✓' if match else '✗'}")

    print(f"\n{'=' * 70}")
    print(f"  SONUÇ: {correct_count}/{n_samples} doğru tahmin")
    print(f"  Doğruluk: {correct_count / n_samples * 100:.1f}%")
    print(f"{'=' * 70}")

    if correct_count == n_samples:
        print("\n[✓] TÜM ŞİFRELİ TAHMİNLER DOĞRU! SIMD bootstrapping başarılı.")
    else:
        print(f"\n[!] {n_samples - correct_count} tahmin uyuşmadı.")


if __name__ == "__main__":
    run_bootstrapped_fhe_batched()

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import time
from desilofhe import Engine

# ─────────────────────────────────────────────────────────────────
def chebyshev_coeffs_step(degree: int, delta: float = 0.05):
    """
    Step fonksiyonu için Chebyshev katsayılarını hesaplar.
    step(x) = 1 if x > 0 else 0
    """
    from scipy.fftpack import dct

    # Grid points
    n = degree + 1
    x = np.cos(np.pi * (np.arange(n) + 0.5) / n)

    # Target function (step at 0)
    # x in [-1, 1]
    y = np.where(x > 0, 1.0, 0.0)

    # DCT to find coefficients
    c = dct(y, type=2, norm='ortho')
    # Normalize for standard Chebyshev definition
    c[0] /= np.sqrt(2)
    c *= (np.sqrt(2.0 / n))
    return c
def eval_chebyshev_plaintext(coeffs, x):
    """Plaintext Chebyshev değerlendirmesi (Validation için)"""
    T = [np.ones_like(x), x]
    for i in range(2, len(coeffs)):
        T.append(2 * x * T[i-1] - T[i-2])

    res = np.zeros_like(x)
    for i, c in enumerate(coeffs):
        res += c * T[i]
    return res
def prepare_data():
    """Iris veriseti üzerinde Logistic Regression eğit."""
    print("[*] Iris veriseti yükleniyor ve model eğitiliyor...")
    iris = load_iris()
    X, y = iris.data, iris.target
    # Binary sınıflandırma: Setosa (0) = 1, Diğerleri = 0
    y_bin = (y == 0).astype(int)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_bin, test_size=0.2, random_state=42
    )
    clf = LogisticRegression(random_state=42)
    clf.fit(X_train, y_train)
    weights = clf.coef_[0]
    bias = clf.intercept_[0]
    train_acc = clf.score(X_train, y_train)
    test_acc = clf.score(X_test, y_test)
    print(f"[✓] Model Eğitildi. Train acc: {train_acc:.4f}, Test acc: {test_acc:.4f}")
    return X_test, y_test, weights, bias, scaler, clf
def run_bootstrapped_fhe():
    X_test, y_test, weights, bias, scaler, clf = prepare_data()
    print("\n" + "=" * 70)
    print("  Bootstrapping: Gerçek FHE Iris Sınıflandırma")
    print("=" * 70)
    # ─────────────────────────────────────────────────────────────────
    # ADIM 0: FHE Engine + Bootstrap Anahtarları
    # ─────────────────────────────────────────────────────────────────
    print("\n[ADIM 0] FHE Engine (bootstrap destekli) başlatılıyor...")
    t0 = time.time()
    # Bootstrapping-enabled motor (Parametreler kütüphane tarafından sabitlenmiştir)
    engine = Engine(use_bootstrap=True)
    print("  → Secret key üretiliyor...")
    sk = engine.create_secret_key()
    print("  → Public key üretiliyor...")
    pk = engine.create_public_key(sk)
    print("  → Relinearization key üretiliyor...")
    rk = engine.create_relinearization_key(sk)
    print("  → Conjugation key üretiliyor...")
    conj_key = engine.create_conjugation_key(sk)
    print("  → Lossy Bootstrap key üretiliyor (bu biraz sürebilir)...")
    t_bk = time.time()
    lbk = engine.create_lossy_bootstrap_key(sk, stage_count=3)
    print(f"  → Bootstrap key hazır. Süre: {time.time() - t_bk:.2f} sn")
    total_setup = time.time() - t0
    print(f"\n[✓] Tüm anahtarlar hazır. Toplam kurulum süresi: {total_setup:.2f} sn")
    print(f"    Slot sayısı: {engine.slot_count}")
    print(f"    Maksimum level: {engine.max_level}")
    # Chebyshev katsayıları
    step_coeffs = chebyshev_coeffs_step(degree=15, delta=0.05)
    # ─────────────────────────────────────────────────────────────────
    # Birden fazla test örneği üzerinde ardışık FHE inference
    # ─────────────────────────────────────────────────────────────────
    n_samples = min(30, len(X_test))
    print(f"\n{'=' * 70}")
    print(f"  {n_samples} test örneği üzerinde şifreli tahmin")
    print(f"{'=' * 70}")
    correct_count = 0
    level = engine.max_level
    for idx in range(n_samples):
        sample = X_test[idx]
        actual = y_test[idx]
        # Plaintext tahmin (doğrulama için)
        logit_pt = np.dot(sample, weights) + bias
        pred_pt = 1 if logit_pt > 0 else 0
        print(f"\n--- Örnek {idx + 1}/{n_samples} ---")
        print(f"  Özellikler: [{', '.join(f'{v:.3f}' for v in sample)}]")
        print(f"  Gerçek sınıf: {actual}  |  Plaintext tahmin: {pred_pt}")
        # ─── ADIM 1: Şifreleme ───
        t_enc = time.time()
        cts = []
        for f_val in sample:
            arr = np.zeros(engine.slot_count)
            arr[0] = f_val
            ct = engine.encrypt(arr, pk)
            cts.append(ct)
        # ─── ADIM 2: Şifreli wx + b ───
        ct_dot = None
        for i, w in enumerate(weights):
            w_arr = np.zeros(engine.slot_count)
            w_arr[0] = w
            pt_w = engine.encode(w_arr, level)
            ct_wx_i = engine.multiply(cts[i], pt_w)
            if ct_dot is None:
                ct_dot = ct_wx_i
            else:
                ct_dot = engine.add(ct_dot, ct_wx_i)
        # Bias ekleme
        b_arr = np.zeros(engine.slot_count)
        b_arr[0] = bias
        pt_b = engine.encode(b_arr, ct_dot.level)
        ct_logit = engine.add(ct_dot, pt_b)
        print(f"  Level (wx+b sonrası): {ct_logit.level}")
        # ─── ADIM 3: Bootstrap (level yenileme) ───
        # Bootstrapping'i GÖSTERMEK için eşiği yüksek tutuyoruz (25 ve altı)
        if ct_logit.level <= 25:
            print(f"  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: {ct_logit.level})...")
            t_bs = time.time()
            ct_logit_intt = engine.intt(ct_logit)
            ct_logit = engine.lossy_bootstrap(
                ct_logit_intt, rk, conj_key, lbk
            )
            print(f"  [BOOTSTRAP] Yeni level: {ct_logit.level}  (süre: {time.time() - t_bs:.2f} sn)")
        # ─── ADIM 4: Normalizasyon + BLEACH Aktivasyonu ───
        # Logit'i [-1, 1] aralığına normalize et
        scale_factor = 20.0
        s_arr = np.zeros(engine.slot_count)
        s_arr[0] = 1.0 / scale_factor
        pt_s = engine.encode(s_arr, ct_logit.level)
        ct_norm = engine.multiply(ct_logit, pt_s)
        print(f"  Level (normalize sonrası): {ct_norm.level}")
        # Level yeterliliğini kontrol et, gerekirse tekrar bootstrap
        if ct_norm.level < 5:
            print(f"  [BOOTSTRAP] Chebyshev öncesi level düşük ({ct_norm.level}), bootstrap...")
            t_bs2 = time.time()
            ct_norm_intt_bs = engine.intt(ct_norm)
            ct_norm = engine.lossy_bootstrap(
                ct_norm_intt_bs, rk, conj_key, lbk
            )
            print(f"  [BOOTSTRAP] Yeni level: {ct_norm.level}  (süre: {time.time() - t_bs2:.2f} sn)")
        # Chebyshev step fonksiyonu
        ct_norm_intt = engine.intt(ct_norm)
        ct_pred = engine.evaluate_chebyshev_polynomial(
            ct_norm_intt, step_coeffs.tolist(), rk
        )
        # ─── ADIM 5: Şifre çözme ───
        result = engine.decrypt(ct_pred, sk)
        fhe_val = result[0]
        fhe_class = int(np.round(fhe_val))
        fhe_class = max(0, min(1, fhe_class))  # 0 veya 1'e clamp
        t_total = time.time() - t_enc
        match = fhe_class == pred_pt
        if match:
            correct_count += 1
        print(f"  FHE ham değer: {fhe_val:.6f}")
        print(f"  FHE tahmin: {fhe_class} {'✓' if match else '✗'}  (süre: {t_total:.2f} sn)")
    # ─── Özet ───
    print(f"\n{'=' * 70}")
    print(f"  SONUÇ: {correct_count}/{n_samples} doğru tahmin")
    print(f"  Doğruluk: {correct_count / n_samples * 100:.1f}%")
    print(f"{'=' * 70}")
    if correct_count == n_samples:
        print("\n[✓] TÜM ŞİFRELİ TAHMİNLER DOĞRU! Bootstrapping başarılı.")
    else:
        print(f"\n[!] {n_samples - correct_count} tahmin uyuşmadı.")
if __name__ == "__main__":
    run_bootstrapped_fhe()

# Output

In [ ]:
[*] Iris veriseti yükleniyor ve model eğitiliyor...
[✓] Model Eğitildi. Train acc: 1.0000, Test acc: 1.0000

======================================================================
  Bootstrapping: Gerçek FHE Iris Sınıflandırma
======================================================================

[ADIM 0] FHE Engine (bootstrap destekli) başlatılıyor...
  → Secret key üretiliyor...
  → Public key üretiliyor...
  → Relinearization key üretiliyor...
  → Conjugation key üretiliyor...
  → Lossy Bootstrap key üretiliyor (bu biraz sürebilir)...
  → Bootstrap key hazır. Süre: 82.52 sn

[✓] Tüm anahtarlar hazır. Toplam kurulum süresi: 86.71 sn
    Slot sayısı: 32768
    Maksimum level: 26

======================================================================
  30 test örneği üzerinde şifreli tahmin
======================================================================

--- Örnek 1/30 ---
  Özellikler: [0.311, -0.592, 0.535, 0.001]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.98 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.126044
  FHE tahmin: 0 ✓  (süre: 42.14 sn)

--- Örnek 2/30 ---
  Özellikler: [-0.174, 1.710, -1.170, -1.184]
  Gerçek sınıf: 1  |  Plaintext tahmin: 1
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.65 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 1.141355
  FHE tahmin: 1 ✓  (süre: 41.52 sn)

--- Örnek 3/30 ---
  Özellikler: [2.250, -1.053, 1.786, 1.449]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.45 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 0.021774
  FHE tahmin: 0 ✓  (süre: 41.06 sn)

--- Örnek 4/30 ---
  Özellikler: [0.190, -0.362, 0.422, 0.396]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.44 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.125001
  FHE tahmin: 0 ✓  (süre: 41.08 sn)

--- Örnek 5/30 ---
  Özellikler: [1.159, -0.592, 0.592, 0.264]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.41 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.034954
  FHE tahmin: 0 ✓  (süre: 41.03 sn)

--- Örnek 6/30 ---
  Özellikler: [-0.537, 0.789, -1.283, -1.052]
  Gerçek sınıf: 1  |  Plaintext tahmin: 1
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.39 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 1.108621
  FHE tahmin: 1 ✓  (süre: 41.02 sn)

--- Örnek 7/30 ---
  Özellikler: [-0.295, -0.362, -0.090, 0.133]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.29 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.087771
  FHE tahmin: 0 ✓  (süre: 40.87 sn)

--- Örnek 8/30 ---
  Özellikler: [1.280, 0.098, 0.763, 1.449]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.32 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 0.052191
  FHE tahmin: 0 ✓  (süre: 40.94 sn)

--- Örnek 9/30 ---
  Özellikler: [0.432, -1.974, 0.422, 0.396]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.33 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 0.020072
  FHE tahmin: 0 ✓  (süre: 40.94 sn)

--- Örnek 10/30 ---
  Özellikler: [-0.053, -0.823, 0.081, 0.001]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.37 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.140329
  FHE tahmin: 0 ✓  (süre: 40.97 sn)

--- Örnek 11/30 ---
  Özellikler: [0.796, 0.328, 0.763, 1.054]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.23 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.031303
  FHE tahmin: 0 ✓  (süre: 40.84 sn)

--- Örnek 12/30 ---
  Özellikler: [-1.264, -0.132, -1.340, -1.447]
  Gerçek sınıf: 1  |  Plaintext tahmin: 1
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.30 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 1.131517
  FHE tahmin: 1 ✓  (süre: 40.88 sn)

--- Örnek 13/30 ---
  Özellikler: [-0.416, 1.019, -1.397, -1.315]
  Gerçek sınıf: 1  |  Plaintext tahmin: 1
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.33 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 1.141557
  FHE tahmin: 1 ✓  (süre: 40.96 sn)

--- Örnek 14/30 ---
  Özellikler: [-1.143, 0.098, -1.283, -1.447]
  Gerçek sınıf: 1  |  Plaintext tahmin: 1
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.30 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 1.133675
  FHE tahmin: 1 ✓  (süre: 40.88 sn)

--- Örnek 15/30 ---
  Özellikler: [-0.901, 1.710, -1.283, -1.184]
  Gerçek sınıf: 1  |  Plaintext tahmin: 1
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.27 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 1.119161
  FHE tahmin: 1 ✓  (süre: 40.86 sn)

--- Örnek 16/30 ---
  Özellikler: [0.553, 0.559, 0.535, 0.527]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.25 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.136825
  FHE tahmin: 0 ✓  (süre: 40.85 sn)

--- Örnek 17/30 ---
  Özellikler: [0.796, -0.132, 1.161, 1.317]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.29 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 0.061233
  FHE tahmin: 0 ✓  (süre: 40.89 sn)

--- Örnek 18/30 ---
  Özellikler: [-0.295, -1.283, 0.081, -0.131]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.23 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.141541
  FHE tahmin: 0 ✓  (süre: 40.86 sn)

--- Örnek 19/30 ---
  Özellikler: [-0.174, -0.592, 0.422, 0.133]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.18 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.140761
  FHE tahmin: 0 ✓  (süre: 40.76 sn)

--- Örnek 20/30 ---
  Özellikler: [0.675, -0.592, 1.047, 1.317]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.32 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 0.067938
  FHE tahmin: 0 ✓  (süre: 40.98 sn)

--- Örnek 21/30 ---
  Özellikler: [-1.385, 0.328, -1.227, -1.315]
  Gerçek sınıf: 1  |  Plaintext tahmin: 1
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.25 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 1.140045
  FHE tahmin: 1 ✓  (süre: 40.84 sn)

--- Örnek 22/30 ---
  Özellikler: [0.311, -0.132, 0.649, 0.791]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.48 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.072372
  FHE tahmin: 0 ✓  (süre: 41.28 sn)

--- Örnek 23/30 ---
  Özellikler: [-1.022, 0.789, -1.227, -1.052]
  Gerçek sınıf: 1  |  Plaintext tahmin: 1
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.24 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 1.133054
  FHE tahmin: 1 ✓  (süre: 40.83 sn)

--- Örnek 24/30 ---
  Özellikler: [0.675, -0.592, 1.047, 1.186]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.20 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 0.062021
  FHE tahmin: 0 ✓  (süre: 40.78 sn)

--- Örnek 25/30 ---
  Özellikler: [2.492, 1.710, 1.502, 1.054]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.21 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 0.051177
  FHE tahmin: 0 ✓  (süre: 40.77 sn)

--- Örnek 26/30 ---
  Özellikler: [1.038, -0.132, 0.820, 1.449]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.07 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 0.056881
  FHE tahmin: 0 ✓  (süre: 40.64 sn)

--- Örnek 27/30 ---
  Özellikler: [1.038, -1.283, 1.161, 0.791]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.19 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 0.077320
  FHE tahmin: 0 ✓  (süre: 40.74 sn)

--- Örnek 28/30 ---
  Özellikler: [1.159, 0.328, 1.217, 1.449]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.22 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 0.065235
  FHE tahmin: 0 ✓  (süre: 40.83 sn)

--- Örnek 29/30 ---
  Özellikler: [-1.264, -0.132, -1.340, -1.184]
  Gerçek sınıf: 1  |  Plaintext tahmin: 1
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.26 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 1.105082
  FHE tahmin: 1 ✓  (süre: 40.86 sn)

--- Örnek 30/30 ---
  Özellikler: [-1.264, 0.098, -1.227, -1.315]
  Gerçek sınıf: 1  |  Plaintext tahmin: 1
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.31 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 1.125287
  FHE tahmin: 1 ✓  (süre: 40.94 sn)

======================================================================
  SONUÇ: 30/30 doğru tahmin
  Doğruluk: 100.0%
======================================================================

[✓] TÜM ŞİFRELİ TAHMİNLER DOĞRU! Bootstrapping başarılı.


In [ ]:
[*] Iris veriseti yükleniyor ve model eğitiliyor...
[✓] Model Eğitildi. Train acc: 1.0000, Test acc: 1.0000

======================================================================
  BLEACH + Bootstrapping: Batch Paketleme ile Optimize FHE
======================================================================

[ADIM 0] FHE Engine (bootstrap destekli) başlatılıyor...
  → Secret key üretiliyor...
  → Public key üretiliyor...
  → Relinearization key üretiliyor...
  → Conjugation key üretiliyor...
  → Lossy Bootstrap key üretiliyor (bu biraz sürebilir)...
  → Bootstrap key hazır. Süre: 82.42 sn

[✓] Tüm anahtarlar hazır. Toplam kurulum süresi: 86.49 sn
    Slot sayısı: 32768
    Maksimum level: 26

======================================================================
  Paketleme: 30 örnek × 4 özellik
  Toplam kullanılan slot: 120 / 32768
  Şifreleme sayısı: 4 ct (önceki: 120 ct)
======================================================================

[ADIM 1] Batch şifreleme...
  → 4 ciphertext ile 30 örnek şifrelendi.
  → Şifreleme süresi: 1.38 sn

[ADIM 2] Şifreli wx + b hesaplanıyor (tüm örnekler paralel)...
  → Level (wx+b sonrası): 25

[ADIM 3] Bootstrap (tek seferlik, tüm örnekler)...
  Mevcut level: 25
  → Yeni level: 13  (süre: 32.16 sn)

[ADIM 4] Normalizasyon + Chebyshev step (tüm örnekler paralel)...
  → Level (normalize sonrası): 12

[ADIM 5] Şifre çözme (tek decrypt, tüm örnekler)...

======================================================================
  Sonuçlar
======================================================================

  Örnek 1/30
    Özellikler: [0.311, -0.592, 0.535, 0.001]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.126044
    FHE tahmin: 0 ✓

  Örnek 2/30
    Özellikler: [-0.174, 1.710, -1.170, -1.184]
    Gerçek sınıf: 1  |  Plaintext tahmin: 1
    FHE ham değer: 1.141355
    FHE tahmin: 1 ✓

  Örnek 3/30
    Özellikler: [2.250, -1.053, 1.786, 1.449]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: 0.021774
    FHE tahmin: 0 ✓

  Örnek 4/30
    Özellikler: [0.190, -0.362, 0.422, 0.396]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.125001
    FHE tahmin: 0 ✓

  Örnek 5/30
    Özellikler: [1.159, -0.592, 0.592, 0.264]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.034954
    FHE tahmin: 0 ✓

  Örnek 6/30
    Özellikler: [-0.537, 0.789, -1.283, -1.052]
    Gerçek sınıf: 1  |  Plaintext tahmin: 1
    FHE ham değer: 1.108621
    FHE tahmin: 1 ✓

  Örnek 7/30
    Özellikler: [-0.295, -0.362, -0.090, 0.133]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.087771
    FHE tahmin: 0 ✓

  Örnek 8/30
    Özellikler: [1.280, 0.098, 0.763, 1.449]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: 0.052191
    FHE tahmin: 0 ✓

  Örnek 9/30
    Özellikler: [0.432, -1.974, 0.422, 0.396]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: 0.020072
    FHE tahmin: 0 ✓

  Örnek 10/30
    Özellikler: [-0.053, -0.823, 0.081, 0.001]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.140329
    FHE tahmin: 0 ✓

  Örnek 11/30
    Özellikler: [0.796, 0.328, 0.763, 1.054]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.031303
    FHE tahmin: 0 ✓

  Örnek 12/30
    Özellikler: [-1.264, -0.132, -1.340, -1.447]
    Gerçek sınıf: 1  |  Plaintext tahmin: 1
    FHE ham değer: 1.131517
    FHE tahmin: 1 ✓

  Örnek 13/30
    Özellikler: [-0.416, 1.019, -1.397, -1.315]
    Gerçek sınıf: 1  |  Plaintext tahmin: 1
    FHE ham değer: 1.141557
    FHE tahmin: 1 ✓

  Örnek 14/30
    Özellikler: [-1.143, 0.098, -1.283, -1.447]
    Gerçek sınıf: 1  |  Plaintext tahmin: 1
    FHE ham değer: 1.133675
    FHE tahmin: 1 ✓

  Örnek 15/30
    Özellikler: [-0.901, 1.710, -1.283, -1.184]
    Gerçek sınıf: 1  |  Plaintext tahmin: 1
    FHE ham değer: 1.119161
    FHE tahmin: 1 ✓

  Örnek 16/30
    Özellikler: [0.553, 0.559, 0.535, 0.527]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.136825
    FHE tahmin: 0 ✓

  Örnek 17/30
    Özellikler: [0.796, -0.132, 1.161, 1.317]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: 0.061233
    FHE tahmin: 0 ✓

  Örnek 18/30
    Özellikler: [-0.295, -1.283, 0.081, -0.131]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.141541
    FHE tahmin: 0 ✓

  Örnek 19/30
    Özellikler: [-0.174, -0.592, 0.422, 0.133]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.140761
    FHE tahmin: 0 ✓

  Örnek 20/30
    Özellikler: [0.675, -0.592, 1.047, 1.317]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: 0.067938
    FHE tahmin: 0 ✓

  Örnek 21/30
    Özellikler: [-1.385, 0.328, -1.227, -1.315]
    Gerçek sınıf: 1  |  Plaintext tahmin: 1
    FHE ham değer: 1.140045
    FHE tahmin: 1 ✓

  Örnek 22/30
    Özellikler: [0.311, -0.132, 0.649, 0.791]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.072372
    FHE tahmin: 0 ✓

  Örnek 23/30
    Özellikler: [-1.022, 0.789, -1.227, -1.052]
    Gerçek sınıf: 1  |  Plaintext tahmin: 1
    FHE ham değer: 1.133054
    FHE tahmin: 1 ✓

  Örnek 24/30
    Özellikler: [0.675, -0.592, 1.047, 1.186]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: 0.062021
    FHE tahmin: 0 ✓

  Örnek 25/30
    Özellikler: [2.492, 1.710, 1.502, 1.054]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: 0.051177
    FHE tahmin: 0 ✓

  Örnek 26/30
    Özellikler: [1.038, -0.132, 0.820, 1.449]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: 0.056881
    FHE tahmin: 0 ✓

  Örnek 27/30
    Özellikler: [1.038, -1.283, 1.161, 0.791]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: 0.077320
    FHE tahmin: 0 ✓

  Örnek 28/30
    Özellikler: [1.159, 0.328, 1.217, 1.449]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: 0.065235
    FHE tahmin: 0 ✓

  Örnek 29/30
    Özellikler: [-1.264, -0.132, -1.340, -1.184]
    Gerçek sınıf: 1  |  Plaintext tahmin: 1
    FHE ham değer: 1.105082
    FHE tahmin: 1 ✓

  Örnek 30/30
    Özellikler: [-1.264, 0.098, -1.227, -1.315]
    Gerçek sınıf: 1  |  Plaintext tahmin: 1
    FHE ham değer: 1.125287
    FHE tahmin: 1 ✓

======================================================================
  SONUÇ: 30/30 doğru tahmin
  Doğruluk: 100.0%
======================================================================

[✓] TÜM ŞİFRELİ TAHMİNLER DOĞRU! SIMD bootstrapping başarılı.


In [ ]:
[*] Iris veriseti yükleniyor ve model eğitiliyor...
[✓] Model Eğitildi. Train acc: 1.0000, Test acc: 1.0000

======================================================================
  BLEACH + Bootstrapping: Gerçek FHE Iris Sınıflandırma
======================================================================

[ADIM 0] FHE Engine (bootstrap destekli) başlatılıyor...
  → Secret key üretiliyor...
  → Public key üretiliyor...
  → Relinearization key üretiliyor...
  → Conjugation key üretiliyor...
  → Lossy Bootstrap key üretiliyor (bu biraz sürebilir)...
  → Bootstrap key hazır. Süre: 100.02 sn

[✓] Tüm anahtarlar hazır. Toplam kurulum süresi: 104.45 sn
    Slot sayısı: 32768
    Maksimum level: 26

======================================================================
  5 test örneği üzerinde şifreli tahmin
======================================================================

--- Örnek 1/5 ---
  Özellikler: [0.311, -0.592, 0.535, 0.001]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 33.05 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.126044
  FHE tahmin: 0 ✓  (süre: 41.89 sn)

--- Örnek 2/5 ---
  Özellikler: [-0.174, 1.710, -1.170, -1.184]
  Gerçek sınıf: 1  |  Plaintext tahmin: 1
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.61 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 1.141355
  FHE tahmin: 1 ✓  (süre: 41.29 sn)

--- Örnek 3/5 ---
  Özellikler: [2.250, -1.053, 1.786, 1.449]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.59 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 0.021774
  FHE tahmin: 0 ✓  (süre: 41.20 sn)

--- Örnek 4/5 ---
  Özellikler: [0.190, -0.362, 0.422, 0.396]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.58 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.125001
  FHE tahmin: 0 ✓  (süre: 41.21 sn)

--- Örnek 5/5 ---
  Özellikler: [1.159, -0.592, 0.592, 0.264]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.60 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.034954
  FHE tahmin: 0 ✓  (süre: 41.22 sn)

======================================================================
  SONUÇ: 5/5 doğru tahmin
  Doğruluk: 100.0%
======================================================================

[✓] TÜM ŞİFRELİ TAHMİNLER DOĞRU! Bootstrapping başarılı.

In [ ]:
[*] Iris veriseti yükleniyor ve model eğitiliyor...
[✓] Model Eğitildi. Train acc: 1.0000, Test acc: 1.0000

======================================================================
  BLEACH + Bootstrapping: Batch Paketleme ile Optimize FHE
======================================================================

[ADIM 0] FHE Engine (bootstrap destekli) başlatılıyor...
  → Secret key üretiliyor...
  → Public key üretiliyor...
  → Relinearization key üretiliyor...
  → Conjugation key üretiliyor...
  → Lossy Bootstrap key üretiliyor (bu biraz sürebilir)...
  → Bootstrap key hazır. Süre: 100.04 sn

[✓] Tüm anahtarlar hazır. Toplam kurulum süresi: 104.38 sn
    Slot sayısı: 32768
    Maksimum level: 26

======================================================================
  Paketleme: 5 örnek × 4 özellik
  Toplam kullanılan slot: 20 / 32768
  Şifreleme sayısı: 4 ct (önceki: 20 ct)
======================================================================

[ADIM 1] Batch şifreleme...
  → 4 ciphertext ile 5 örnek şifrelendi.
  → Şifreleme süresi: 1.40 sn

[ADIM 2] Şifreli wx + b hesaplanıyor (tüm örnekler paralel)...
  → Level (wx+b sonrası): 25

[ADIM 3] Bootstrap (tek seferlik, tüm örnekler)...
  Mevcut level: 25
  → Yeni level: 13  (süre: 32.71 sn)

[ADIM 4] Normalizasyon + Chebyshev step (tüm örnekler paralel)...
  → Level (normalize sonrası): 12

[ADIM 5] Şifre çözme (tek decrypt, tüm örnekler)...

======================================================================
  Sonuçlar
======================================================================

  Örnek 1/5
    Özellikler: [0.311, -0.592, 0.535, 0.001]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.126044
    FHE tahmin: 0 ✓

  Örnek 2/5
    Özellikler: [-0.174, 1.710, -1.170, -1.184]
    Gerçek sınıf: 1  |  Plaintext tahmin: 1
    FHE ham değer: 1.141355
    FHE tahmin: 1 ✓

  Örnek 3/5
    Özellikler: [2.250, -1.053, 1.786, 1.449]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: 0.021774
    FHE tahmin: 0 ✓

  Örnek 4/5
    Özellikler: [0.190, -0.362, 0.422, 0.396]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.125001
    FHE tahmin: 0 ✓

  Örnek 5/5
    Özellikler: [1.159, -0.592, 0.592, 0.264]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.034954
    FHE tahmin: 0 ✓

======================================================================
  SONUÇ: 5/5 doğru tahmin
  Doğruluk: 100.0%
======================================================================

[✓] TÜM ŞİFRELİ TAHMİNLER DOĞRU! Batch bootstrapping başarılı.

**Code OpenFHE**

In [ ]:
cpp_code = r"""
#include "openfhe.h"
#include <complex>
#include <iostream>
#include <iomanip>
#include <vector>
#include <chrono>
using namespace lbcrypto;
using namespace std::chrono;

using Clock = high_resolution_clock;
using Ms    = duration<double, std::milli>;

void IrisFunctionalBootstrapping() {

    CCParams<CryptoContextCKKSRNS> parameters;
    parameters.SetSecretKeyDist(SPARSE_TERNARY);
    parameters.SetSecurityLevel(HEStd_NotSet);
    parameters.SetRingDim(1 << 13);
    parameters.SetNumLargeDigits(3);
    parameters.SetKeySwitchTechnique(HYBRID);
    parameters.SetScalingModSize(45);
    parameters.SetScalingTechnique(FIXEDMANUAL);
    parameters.SetFirstModSize(46);

    std::vector<uint32_t> levelBudget = {4, 2};
    std::vector<uint32_t> bsgsDim     = {0, 0};
    int bits = 5;
    int p    = 32;

    usint depth = levelBudget[0] + levelBudget[1] + 1 + 5 + 4 + bits + 2;
    parameters.SetMultiplicativeDepth(depth);

    std::cout << "\n========================================" << std::endl;
    std::cout << "  SIMD CKKS — Anahtar Kurulum" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << "  p (modul)           : " << p << "  (guvenli aralik: (-16, +16))" << std::endl;
    std::cout << "  bits                : " << bits << std::endl;
    std::cout << "  depth               : " << depth << std::endl;

    auto t_setup_start = Clock::now();

    CryptoContext<DCRTPoly> cc = GenCryptoContext(parameters);
    cc->Enable(PKE);
    cc->Enable(KEYSWITCH);
    cc->Enable(LEVELEDSHE);
    cc->Enable(ADVANCEDSHE);
    cc->Enable(FHE);
    cc->Enable(FBTS);

    int numSlots = cc->GetRingDimension() / 2;
    cc->EvalFuncBootstrapSetup(levelBudget, bsgsDim, numSlots, bits);

    auto t_keygen_start = Clock::now();
    auto keyPair = cc->KeyGen();
    cc->EvalMultKeyGen(keyPair.secretKey);
    auto t_keygen_end = Clock::now();

    auto t_bskey_start = Clock::now();
    cc->EvalBootstrapKeyGen(keyPair.secretKey, numSlots);
    auto t_bskey_end = Clock::now();

    // Rotasyon anahtarlari: iç çarpim toplamı için (önceki kodla aynı)
    cc->EvalRotateKeyGen(keyPair.secretKey, {1, 2});

    auto t_setup_end = Clock::now();

    double t_keygen = duration_cast<Ms>(t_keygen_end - t_keygen_start).count();
    double t_bskey  = duration_cast<Ms>(t_bskey_end  - t_bskey_start).count();
    double t_setup  = duration_cast<Ms>(t_setup_end  - t_setup_start).count();

    std::cout << "  Ring boyutu (n)     : " << cc->GetRingDimension() << std::endl;
    std::cout << "  Slot sayisi         : " << numSlots << std::endl;
    std::cout << "  Carpim derinligi    : " << depth << std::endl;
    std::cout << "  Key uretimi         : " << std::fixed << std::setprecision(2) << t_keygen << " ms" << std::endl;
    std::cout << "  Bootstrap key       : " << std::fixed << std::setprecision(2) << t_bskey  << " ms" << std::endl;
    std::cout << "  Toplam kurulum      : " << std::fixed << std::setprecision(2) << t_setup  << " ms" << std::endl;

    // ── Veri ──────────────────────────────────────────────────────────────
    // 5 ornek x 4 ozellik
    std::vector<std::vector<double>> all_features = {
        { 0.311, -0.592,  0.535,  0.001},
        {-0.174,  1.710, -1.170, -1.184},
        { 2.250, -1.053,  1.786,  1.449},
        { 0.190, -0.362,  0.422,  0.396},
        { 1.159, -0.592,  0.592,  0.264}
    };
    std::vector<int> true_labels = {0, 1, 0, 0, 0};
    std::vector<double> weights  = {-1.018, 1.173, -1.675, -1.536};
    double bias = -2.4196;

    int n_samples  = (int)all_features.size();
    int n_features = 4;

    // LUT fonksiyonu — p=32 icin esik 16
    auto f_lut = [](double x) -> double {
        return (x < 16.0) ? 1.0 : 0.0;
    };

    usint targetLevel = depth - levelBudget[1];

    // ── Plaintext doğrulama ───────────────────────────────────────────────
    std::cout << "\n  Plaintext logitler (dogrulama):" << std::endl;
    std::vector<int> pred_pt_all(n_samples);
    for (int i = 0; i < n_samples; i++) {
        double logit_pt = bias;
        for (int j = 0; j < n_features; j++) logit_pt += weights[j] * all_features[i][j];
        pred_pt_all[i] = (logit_pt > 0.0) ? 1 : 0;
        bool in_range = (logit_pt > -16.0 && logit_pt < 16.0);
        std::cout << "    Ornek " << (i+1) << ": logit=" << std::fixed << std::setprecision(4)
                  << logit_pt << (in_range ? "  [OK]" : "  [ARALIK DISI]")
                  << "  pred=" << pred_pt_all[i] << std::endl;
    }

    std::cout << "\n========================================" << std::endl;
    std::cout << "  SIMD Batch Sifreli Cikarim" << std::endl;
    std::cout << "  Strateji: ct_feat[j][slot_i] = ornek_i ozellik_j" << std::endl;
    std::cout << "  " << n_samples << " ornek PARALEL, 1 bootstrap" << std::endl;
    std::cout << "========================================" << std::endl;

    // ── ADIM 1: SIMD Batch Sifreleme ──────────────────────────────────────
    // ct_feat[j]: tum orneklerin j. ozelligi tek bir ciphertext'e
    // slot[i] = all_features[i][j]
    auto t_enc_start = Clock::now();

    std::vector<Ciphertext<DCRTPoly>> ct_feat(n_features);
    for (int j = 0; j < n_features; j++) {
        std::vector<std::complex<double>> slot_vec(numSlots, {0.0, 0.0});
        for (int i = 0; i < n_samples; i++) {
            slot_vec[i] = {all_features[i][j], 0.0};
        }
        Plaintext ptxt = cc->MakeCKKSPackedPlaintext(slot_vec, 1, 0, nullptr, numSlots);
        ct_feat[j] = cc->Encrypt(keyPair.publicKey, ptxt);
    }

    auto t_enc_end = Clock::now();
    double dt_enc = duration_cast<Ms>(t_enc_end - t_enc_start).count();
    std::cout << "\n[1] SIMD Batch Sifreleme   : " << std::fixed << std::setprecision(2)
              << dt_enc << " ms  (" << n_features << " ct, " << n_samples << " ornek paralel)" << std::endl;

    // ── ADIM 2: SIMD wx — her agirlik tum slot'lara yayilir ───────────────
    // w[j] tum slot'larda ayni deger → ct_feat[j] ile carpilinca
    // slot[i] = w[j] * ornek_i_ozellik_j
    auto t_mult_start = Clock::now();

    Ciphertext<DCRTPoly> ct_dot;
    bool first = true;
    for (int j = 0; j < n_features; j++) {
        // w[j] tum slot'lara yayilmis plaintext
        std::vector<std::complex<double>> w_vec(numSlots, {weights[j], 0.0});
        Plaintext ptxt_w = cc->MakeCKKSPackedPlaintext(w_vec, 1, 0, nullptr, numSlots);

        auto ct_wx_j = cc->EvalMult(ct_feat[j], ptxt_w);
        cc->RescaleInPlace(ct_wx_j);

        if (first) { ct_dot = ct_wx_j; first = false; }
        else       { ct_dot = cc->EvalAdd(ct_dot, ct_wx_j); }
    }
    // ct_dot[i] = sum_j(w[j] * x[i][j]) = w^T x[i]  tum ornekler icin

    auto t_mult_end = Clock::now();
    double dt_mult = duration_cast<Ms>(t_mult_end - t_mult_start).count();
    std::cout << "[2] SIMD EvalMult+Add      : " << std::fixed << std::setprecision(2)
              << dt_mult << " ms  (tum ornekler paralel)" << std::endl;

    // ── ADIM 3: Bias — tum slot'lara ayni deger ───────────────────────────
    // bias skaler olarak tum slot'lara eklenir
    auto t_bias_start = Clock::now();
    auto ct_logit = cc->EvalAdd(ct_dot, bias);
    auto t_bias_end = Clock::now();
    double dt_bias = duration_cast<Ms>(t_bias_end - t_bias_start).count();
    std::cout << "[3] Bias ekleme (SIMD)     : " << std::fixed << std::setprecision(2)
              << dt_bias << " ms  (Level: " << ct_logit->GetLevel() << ")" << std::endl;

    // ── ADIM 4: Level hizalama ─────────────────────────────────────────────
    auto t_align_start = Clock::now();
    while (ct_logit->GetLevel() < targetLevel)
        cc->LevelReduceInPlace(ct_logit, nullptr);
    auto t_align_end = Clock::now();
    double dt_align = duration_cast<Ms>(t_align_end - t_align_start).count();
    std::cout << "[4] Level hizalama         : " << std::fixed << std::setprecision(2)
              << dt_align << " ms  (Level: " << ct_logit->GetLevel() << "/" << targetLevel << ")" << std::endl;

    // ── ADIM 5: TEK Bootstrap — 5 ornek paralel ───────────────────────────
    // EvalFuncBootstrap ct_logit uzerinde calisir
    // Her slot[i] bagimsiz olarak f(logit_i) hesaplanir — SIMD
    auto t_bs_start = Clock::now();
    auto ct_result = cc->EvalFuncBootstrap(ct_logit, f_lut, p, 1);
    auto t_bs_end = Clock::now();
    double dt_bs = duration_cast<Ms>(t_bs_end - t_bs_start).count();
    std::cout << "[5] EvalFuncBootstrap(SIMD): " << std::fixed << std::setprecision(2)
              << dt_bs << " ms  (1 kez, " << n_samples << " ornek paralel)" << std::endl;

    // ── ADIM 6: Tek Decrypt — tum sonuclar ────────────────────────────────
    auto t_dec_start = Clock::now();
    Plaintext pt_result;
    cc->Decrypt(keyPair.secretKey, ct_result, &pt_result);
    pt_result->SetLength(n_samples);
    auto t_dec_end = Clock::now();
    double dt_dec = duration_cast<Ms>(t_dec_end - t_dec_start).count();
    std::cout << "[6] Sifre cozme (tek)      : " << std::fixed << std::setprecision(2)
              << dt_dec << " ms" << std::endl;

    // ── Sonuclari ayikla ──────────────────────────────────────────────────
    auto fhe_values = pt_result->GetCKKSPackedValue();

    std::cout << "\n========================================" << std::endl;
    std::cout << "  Sonuclar  (slot[i] = ornek i)" << std::endl;
    std::cout << "========================================" << std::endl;

    int correct = 0;
    double total_infer = dt_enc + dt_mult + dt_bias + dt_align + dt_bs + dt_dec;

    for (int i = 0; i < n_samples; i++) {
        double fhe_val = fhe_values[i].real();
        int fhe_pred   = (fhe_val > 0.5) ? 1 : 0;
        bool match     = (fhe_pred == pred_pt_all[i]);
        if (match) correct++;

        std::cout << "  Ornek " << (i+1) << ": "
                  << "logit=" << std::fixed << std::setprecision(4) << (bias + [&](){
                        double s = 0;
                        for (int j = 0; j < n_features; j++) s += weights[j]*all_features[i][j];
                        return s;
                     }())
                  << "  FHE=" << std::fixed << std::setprecision(6) << fhe_val
                  << "  pred=" << fhe_pred
                  << "  gercek=" << true_labels[i]
                  << (match ? "  [OK]" : "  [HATA]") << std::endl;
    }

    std::cout << "\n========================================" << std::endl;
    std::cout << "  HESAPLAMA PROFILI  (SIMD, " << n_samples << " ornek)" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << std::fixed << std::setprecision(2);
    std::cout << "  Anahtar kurulum (toplam) : " << t_setup   << " ms" << std::endl;
    std::cout << "    -- Key uretimi         : " << t_keygen  << " ms" << std::endl;
    std::cout << "    -- Bootstrap key       : " << t_bskey   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Sifreli cikarim (toplam) : " << total_infer << " ms" << std::endl;
    std::cout << "    -- SIMD Sifreleme      : " << dt_enc   << " ms  (" << n_features << " ct)" << std::endl;
    std::cout << "    -- SIMD EvalMult+Add   : " << dt_mult  << " ms" << std::endl;
    std::cout << "    -- Bias ekleme         : " << dt_bias  << " ms" << std::endl;
    std::cout << "    -- Level hizalama      : " << dt_align << " ms" << std::endl;
    std::cout << "    -- EvalFuncBootstrap   : " << dt_bs    << " ms  (1 kez, " << n_samples << " paralel)" << std::endl;
    std::cout << "    -- Sifre cozme         : " << dt_dec   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Ornek basi amortize      : " << (total_infer / n_samples) << " ms" << std::endl;
    std::cout << "  Bootstrap orani          : " << (100.0 * dt_bs / total_infer) << " %" << std::endl;
    std::cout << "  ================================" << std::endl;
    std::cout << "  SONUC: " << correct << "/" << n_samples
              << "  Dogruluk: " << (100.0 * correct / n_samples) << " %" << std::endl;
    std::cout << "========================================\n" << std::endl;
}

int main() {
    IrisFunctionalBootstrapping();
    return 0;
}
"""

with open("/content/sec-ckks/src/pke/examples/iris_functional.cpp", "w") as f:
    f.write(cpp_code)

print("Dosya yazildi.")

In [ ]:
cpp_code = r"""
#include "openfhe.h"
#include <complex>
#include <iostream>
#include <iomanip>
#include <vector>
#include <chrono>
using namespace lbcrypto;
using namespace std::chrono;

using Clock = high_resolution_clock;
using Ms    = duration<double, std::milli>;

void IrisFunctionalBootstrapping() {

    CCParams<CryptoContextCKKSRNS> parameters;
    parameters.SetSecretKeyDist(SPARSE_TERNARY);
    parameters.SetSecurityLevel(HEStd_NotSet);
    parameters.SetRingDim(1 << 13);
    parameters.SetNumLargeDigits(3);
    parameters.SetKeySwitchTechnique(HYBRID);
    parameters.SetScalingModSize(45);
    parameters.SetScalingTechnique(FIXEDMANUAL);
    parameters.SetFirstModSize(46);

    std::vector<uint32_t> levelBudget = {4, 2};
    std::vector<uint32_t> bsgsDim     = {0, 0};
    int bits = 5;   // 2^5 = 32 = p  (p=16 ile logit=-11.16 sinir disina cikiyordu)
    int p    = 32;  // guvenli aralik: (-16, +16)

    usint depth = levelBudget[0] + levelBudget[1] + 1 + 5 + 4 + bits + 2;
    parameters.SetMultiplicativeDepth(depth);

    std::cout << "\n========================================" << std::endl;
    std::cout << "  Anahtar Kurulum" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << "  p (modul)           : " << p << "  (guvenli aralik: (-16, +16))" << std::endl;
    std::cout << "  bits                : " << bits << std::endl;
    std::cout << "  depth               : " << depth << std::endl;

    auto t_setup_start = Clock::now();

    CryptoContext<DCRTPoly> cc = GenCryptoContext(parameters);
    cc->Enable(PKE);
    cc->Enable(KEYSWITCH);
    cc->Enable(LEVELEDSHE);
    cc->Enable(ADVANCEDSHE);
    cc->Enable(FHE);
    cc->Enable(FBTS);

    int numSlots = cc->GetRingDimension() / 2;
    cc->EvalFuncBootstrapSetup(levelBudget, bsgsDim, numSlots, bits);

    auto t_keygen_start = Clock::now();
    auto keyPair = cc->KeyGen();
    cc->EvalMultKeyGen(keyPair.secretKey);
    auto t_keygen_end = Clock::now();

    auto t_bskey_start = Clock::now();
    cc->EvalBootstrapKeyGen(keyPair.secretKey, numSlots);
    auto t_bskey_end = Clock::now();

    cc->EvalRotateKeyGen(keyPair.secretKey, {1, 2});

    auto t_setup_end = Clock::now();

    double t_keygen = duration_cast<Ms>(t_keygen_end - t_keygen_start).count();
    double t_bskey  = duration_cast<Ms>(t_bskey_end  - t_bskey_start).count();
    double t_setup  = duration_cast<Ms>(t_setup_end  - t_setup_start).count();

    std::cout << "  Ring boyutu (n)     : " << cc->GetRingDimension() << std::endl;
    std::cout << "  Slot sayisi         : " << numSlots << std::endl;
    std::cout << "  Carpim derinligi    : " << depth << std::endl;
    std::cout << "  Key uretimi         : " << std::fixed << std::setprecision(2) << t_keygen << " ms" << std::endl;
    std::cout << "  Bootstrap key       : " << std::fixed << std::setprecision(2) << t_bskey  << " ms" << std::endl;
    std::cout << "  Toplam kurulum      : " << std::fixed << std::setprecision(2) << t_setup  << " ms" << std::endl;

    std::vector<std::vector<double>> all_features = {
        { 0.311, -0.592,  0.535,  0.001},
        {-0.174,  1.710, -1.170, -1.184},
        { 2.250, -1.053,  1.786,  1.449},
        { 0.190, -0.362,  0.422,  0.396},
        { 1.159, -0.592,  0.592,  0.264}
    };
    std::vector<int> true_labels = {0, 1, 0, 0, 0};
    std::vector<double> weights  = {-1.018, 1.173, -1.675, -1.536};
    double bias = -2.4196;

    // p=32: z < 0 → z mod 32 ∈ (16, 32) → f=0 (sinif 0)
    //       z > 0 → z mod 32 ∈ ( 0, 16) → f=1 (sinif 1)
    auto f_lut = [](double x) -> double {
        return (x < 16.0) ? 1.0 : 0.0;
    };

    usint targetLevel = depth - levelBudget[1];

    int n_samples  = (int)all_features.size();
    int n_features = 4;

    double total_enc   = 0.0;
    double total_mult  = 0.0;
    double total_rot   = 0.0;
    double total_bias  = 0.0;
    double total_align = 0.0;
    double total_bs    = 0.0;
    double total_dec   = 0.0;
    double total_infer = 0.0;

    std::cout << "\n========================================" << std::endl;
    std::cout << "  Sifreli Cikarim  (" << n_samples << " ornek)" << std::endl;
    std::cout << "========================================" << std::endl;

    int correct = 0;

    for (int idx = 0; idx < n_samples; idx++) {

        auto t_infer_start = Clock::now();

        const auto& feat = all_features[idx];
        int y_true = true_labels[idx];

        double logit_pt = bias;
        for (int j = 0; j < n_features; j++) logit_pt += weights[j] * feat[j];
        int pred_pt = (logit_pt > 0.0) ? 1 : 0;

        // Modul kisit kontrolu
        bool in_range = (logit_pt > -16.0 && logit_pt < 16.0);

        std::cout << "\n--- Ornek " << (idx+1) << "/" << n_samples << " ---" << std::endl;
        std::cout << "  Ozellikler     : [";
        for (int j = 0; j < n_features; j++)
            std::cout << std::fixed << std::setprecision(3) << feat[j] << (j<3?", ":"");
        std::cout << "]" << std::endl;
        std::cout << "  Gercek sinif   : " << y_true << std::endl;
        std::cout << "  Plaintext logit: " << std::fixed << std::setprecision(4) << logit_pt
                  << (in_range ? "  [aralik OK]" : "  [ARALIK DISI - HATA BEKLENIR]") << std::endl;
        std::cout << "  Plaintext pred : " << pred_pt << std::endl;

        auto t_enc_start = Clock::now();
        std::vector<std::complex<double>> f_vec(numSlots, 0);
        for (int j = 0; j < n_features; j++) f_vec[j] = {feat[j], 0.0};
        Plaintext ptxtF = cc->MakeCKKSPackedPlaintext(f_vec, 1, 0, nullptr, numSlots);
        auto ctxtFeatures = cc->Encrypt(keyPair.publicKey, ptxtF);
        auto t_enc_end = Clock::now();
        double dt_enc = duration_cast<Ms>(t_enc_end - t_enc_start).count();
        total_enc += dt_enc;
        std::cout << "  [1] Sifreleme        : " << std::fixed << std::setprecision(2) << dt_enc << " ms" << std::endl;

        auto t_mult_start = Clock::now();
        std::vector<std::complex<double>> w_vec(numSlots, 0);
        for (int j = 0; j < n_features; j++) w_vec[j] = {weights[j], 0.0};
        Plaintext ptxtW = cc->MakeCKKSPackedPlaintext(w_vec, 1, 0, nullptr, numSlots);
        auto ctxtMult = cc->EvalMult(ctxtFeatures, ptxtW);
        cc->RescaleInPlace(ctxtMult);
        auto t_mult_end = Clock::now();
        double dt_mult = duration_cast<Ms>(t_mult_end - t_mult_start).count();
        total_mult += dt_mult;
        std::cout << "  [2] EvalMult+Rescale : " << std::fixed << std::setprecision(2) << dt_mult << " ms" << std::endl;

        auto t_rot_start = Clock::now();
        auto ctxtSum = ctxtMult;
        for (int i = 0; i < 2; i++)
            ctxtSum = cc->EvalAdd(ctxtSum, cc->EvalAtIndex(ctxtSum, 1 << i));
        auto t_rot_end = Clock::now();
        double dt_rot = duration_cast<Ms>(t_rot_end - t_rot_start).count();
        total_rot += dt_rot;
        std::cout << "  [3] Rotasyon toplami : " << std::fixed << std::setprecision(2) << dt_rot << " ms" << std::endl;

        auto t_bias_start = Clock::now();
        auto ctxtLogit = cc->EvalAdd(ctxtSum, bias);
        auto t_bias_end = Clock::now();
        double dt_bias = duration_cast<Ms>(t_bias_end - t_bias_start).count();
        total_bias += dt_bias;
        std::cout << "  [4] Bias ekleme      : " << std::fixed << std::setprecision(2) << dt_bias << " ms"
                  << "  (Level: " << ctxtLogit->GetLevel() << ")" << std::endl;

        auto t_align_start = Clock::now();
        while (ctxtLogit->GetLevel() < targetLevel)
            cc->LevelReduceInPlace(ctxtLogit, nullptr);
        auto t_align_end = Clock::now();
        double dt_align = duration_cast<Ms>(t_align_end - t_align_start).count();
        total_align += dt_align;
        std::cout << "  [5] Level hizalama   : " << std::fixed << std::setprecision(2) << dt_align << " ms"
                  << "  (Level: " << ctxtLogit->GetLevel() << "/" << targetLevel << ")" << std::endl;

        auto t_bs_start = Clock::now();
        auto ctxtResult = cc->EvalFuncBootstrap(ctxtLogit, f_lut, p, 1);
        auto t_bs_end = Clock::now();
        double dt_bs = duration_cast<Ms>(t_bs_end - t_bs_start).count();
        total_bs += dt_bs;
        std::cout << "  [6] EvalFuncBootstrap: " << std::fixed << std::setprecision(2) << dt_bs << " ms" << std::endl;

        auto t_dec_start = Clock::now();
        Plaintext result;
        cc->Decrypt(keyPair.secretKey, ctxtResult, &result);
        result->SetLength(1);
        auto t_dec_end = Clock::now();
        double dt_dec = duration_cast<Ms>(t_dec_end - t_dec_start).count();
        total_dec += dt_dec;

        double fhe_val = result->GetCKKSPackedValue()[0].real();
        int fhe_pred   = (fhe_val > 0.5) ? 1 : 0;
        bool match     = (fhe_pred == pred_pt);
        if (match) correct++;

        auto t_infer_end = Clock::now();
        double dt_infer = duration_cast<Ms>(t_infer_end - t_infer_start).count();
        total_infer += dt_infer;

        std::cout << "  [7] Sifre cozme      : " << std::fixed << std::setprecision(2) << dt_dec << " ms" << std::endl;
        std::cout << "  --------------------------------" << std::endl;
        std::cout << "  FHE ham deger   : " << std::fixed << std::setprecision(6) << fhe_val << std::endl;
        std::cout << "  FHE tahmin      : " << fhe_pred << (match ? "  [OK]" : "  [HATA]") << std::endl;
        std::cout << "  Ornek suresi    : " << std::fixed << std::setprecision(2) << dt_infer << " ms" << std::endl;
    }

    std::cout << "\n========================================" << std::endl;
    std::cout << "  HESAPLAMA PROFILI  (" << n_samples << " ornek toplam)" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << std::fixed << std::setprecision(2);
    std::cout << "  Anahtar kurulum (toplam) : " << t_setup   << " ms" << std::endl;
    std::cout << "    -- Key uretimi         : " << t_keygen  << " ms" << std::endl;
    std::cout << "    -- Bootstrap key       : " << t_bskey   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Sifreli cikarim (toplam) : " << total_infer << " ms" << std::endl;
    std::cout << "    -- Sifreleme           : " << total_enc   << " ms" << std::endl;
    std::cout << "    -- EvalMult + Rescale  : " << total_mult  << " ms" << std::endl;
    std::cout << "    -- Rotasyon toplami    : " << total_rot   << " ms" << std::endl;
    std::cout << "    -- Bias ekleme         : " << total_bias  << " ms" << std::endl;
    std::cout << "    -- Level hizalama      : " << total_align << " ms" << std::endl;
    std::cout << "    -- EvalFuncBootstrap   : " << total_bs    << " ms" << std::endl;
    std::cout << "    -- Sifre cozme         : " << total_dec   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Ornek basi ortalama      : " << (total_infer / n_samples) << " ms" << std::endl;
    std::cout << "  Bootstrap orani          : " << (100.0 * total_bs / total_infer) << " %" << std::endl;
    std::cout << "  ================================" << std::endl;
    std::cout << "  SONUC: " << correct << "/" << n_samples
              << "  Dogruluk: " << (100.0 * correct / n_samples) << " %" << std::endl;
    std::cout << "========================================\n" << std::endl;
}

int main() {
    IrisFunctionalBootstrapping();
    return 0;
}
"""

with open("/content/sec-ckks/src/pke/examples/iris_functional.cpp", "w") as f:
    f.write(cpp_code)

print("Dosya yazildi.")

30 features

In [ ]:
cpp_code = r"""
#include "openfhe.h"
#include <complex>
#include <iostream>
#include <iomanip>
#include <vector>
#include <chrono>
using namespace lbcrypto;
using namespace std::chrono;

using Clock = high_resolution_clock;
using Ms    = duration<double, std::milli>;

void IrisFunctionalBootstrapping() {

    CCParams<CryptoContextCKKSRNS> parameters;
    parameters.SetSecretKeyDist(SPARSE_TERNARY);
    parameters.SetSecurityLevel(HEStd_128_classic);
    parameters.SetRingDim(1 << 13);
    parameters.SetNumLargeDigits(3);
    parameters.SetKeySwitchTechnique(HYBRID);
    parameters.SetScalingModSize(45);
    parameters.SetScalingTechnique(FIXEDMANUAL);
    parameters.SetFirstModSize(46);

    std::vector<uint32_t> levelBudget = {4, 2};
    std::vector<uint32_t> bsgsDim     = {0, 0};
    int bits = 5;
    int p    = 32;

    usint depth = levelBudget[0] + levelBudget[1] + 1 + 5 + 4 + bits + 2;
    parameters.SetMultiplicativeDepth(depth);

    std::cout << "\n========================================" << std::endl;
    std::cout << "  SIMD CKKS — Anahtar Kurulum" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << "  p (modul)           : " << p << "  (guvenli aralik: (-16, +16))" << std::endl;
    std::cout << "  bits                : " << bits << std::endl;
    std::cout << "  depth               : " << depth << std::endl;

    auto t_setup_start = Clock::now();

    CryptoContext<DCRTPoly> cc = GenCryptoContext(parameters);
    cc->Enable(PKE);
    cc->Enable(KEYSWITCH);
    cc->Enable(LEVELEDSHE);
    cc->Enable(ADVANCEDSHE);
    cc->Enable(FHE);
    cc->Enable(FBTS);

    int numSlots = cc->GetRingDimension() / 2;
    cc->EvalFuncBootstrapSetup(levelBudget, bsgsDim, numSlots, bits);

    auto t_keygen_start = Clock::now();
    auto keyPair = cc->KeyGen();
    cc->EvalMultKeyGen(keyPair.secretKey);
    auto t_keygen_end = Clock::now();

    auto t_bskey_start = Clock::now();
    cc->EvalBootstrapKeyGen(keyPair.secretKey, numSlots);
    auto t_bskey_end = Clock::now();

    // Rotasyon anahtarlari: iç çarpim toplamı için (önceki kodla aynı)
    cc->EvalRotateKeyGen(keyPair.secretKey, {1, 2});

    auto t_setup_end = Clock::now();

    double t_keygen = duration_cast<Ms>(t_keygen_end - t_keygen_start).count();
    double t_bskey  = duration_cast<Ms>(t_bskey_end  - t_bskey_start).count();
    double t_setup  = duration_cast<Ms>(t_setup_end  - t_setup_start).count();

    std::cout << "  Ring boyutu (n)     : " << cc->GetRingDimension() << std::endl;
    std::cout << "  Slot sayisi         : " << numSlots << std::endl;
    std::cout << "  Carpim derinligi    : " << depth << std::endl;
    std::cout << "  Key uretimi         : " << std::fixed << std::setprecision(2) << t_keygen << " ms" << std::endl;
    std::cout << "  Bootstrap key       : " << std::fixed << std::setprecision(2) << t_bskey  << " ms" << std::endl;
    std::cout << "  Toplam kurulum      : " << std::fixed << std::setprecision(2) << t_setup  << " ms" << std::endl;

    // ── Veri ──────────────────────────────────────────────────────────────
    // 30 ornek x 4 ozellik
    std::vector<std::vector<double>> all_features = {
        {  0.3110,  -0.5924,   0.5354,   0.0009},
        { -0.1737,   1.7096,  -1.1697,  -1.1838},
        {  2.2497,  -1.0528,   1.7858,   1.4488},
        {  0.1898,  -0.3622,   0.4217,   0.3958},
        {  1.1592,  -0.5924,   0.5922,   0.2641},
        { -0.5372,   0.7888,  -1.2834,  -1.0522},
        { -0.2948,  -0.3622,  -0.0898,   0.1325},
        {  1.2803,   0.0982,   0.7628,   1.4488},
        {  0.4322,  -1.9736,   0.4217,   0.3958},
        { -0.0525,  -0.8226,   0.0807,   0.0009},
        {  0.7957,   0.3284,   0.7628,   1.0539},
        { -1.2642,  -0.1320,  -1.3402,  -1.4471},
        { -0.4160,   1.0190,  -1.3971,  -1.3154},
        { -1.1430,   0.0982,  -1.2834,  -1.4471},
        { -0.9007,   1.7096,  -1.2834,  -1.1838},
        {  0.5533,   0.5586,   0.5354,   0.5274},
        {  0.7957,  -0.1320,   1.1606,   1.3172},
        { -0.2948,  -1.2830,   0.0807,  -0.1308},
        { -0.1737,  -0.5924,   0.4217,   0.1325},
        {  0.6745,  -0.5924,   1.0469,   1.3172},
        { -1.3854,   0.3284,  -1.2266,  -1.3154},
        {  0.3110,  -0.1320,   0.6491,   0.7907},
        { -1.0218,   0.7888,  -1.2266,  -1.0522},
        {  0.6745,  -0.5924,   1.0469,   1.1856},
        {  2.4920,   1.7096,   1.5016,   1.0539},
        {  1.0380,  -0.1320,   0.8196,   1.4488},
        {  1.0380,  -1.2830,   1.1606,   0.7907},
        {  1.1592,   0.3284,   1.2175,   1.4488},
        { -1.2642,  -0.1320,  -1.3402,  -1.1838},
        { -1.2642,   0.0982,  -1.2266,  -1.3154}
    };
    std::vector<int> true_labels = {0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1};
    std::vector<double> weights  = {-1.0183, 1.1725, -1.6753, -1.5359};
    double bias = -2.4196;

    int n_samples  = (int)all_features.size();
    int n_features = 4;

    // LUT fonksiyonu — p=32 icin esik 16
    auto f_lut = [](double x) -> double {
        return (x < 16.0) ? 1.0 : 0.0;
    };

    usint targetLevel = depth - levelBudget[1];

    // ── Plaintext doğrulama ───────────────────────────────────────────────
    std::cout << "\n  Plaintext logitler (dogrulama):" << std::endl;
    std::vector<int> pred_pt_all(n_samples);
    for (int i = 0; i < n_samples; i++) {
        double logit_pt = bias;
        for (int j = 0; j < n_features; j++) logit_pt += weights[j] * all_features[i][j];
        pred_pt_all[i] = (logit_pt > 0.0) ? 1 : 0;
        bool in_range = (logit_pt > -16.0 && logit_pt < 16.0);
        std::cout << "    Ornek " << (i+1) << ": logit=" << std::fixed << std::setprecision(4)
                  << logit_pt << (in_range ? "  [OK]" : "  [ARALIK DISI]")
                  << "  pred=" << pred_pt_all[i] << std::endl;
    }

    std::cout << "\n========================================" << std::endl;
    std::cout << "  SIMD Batch Sifreli Cikarim" << std::endl;
    std::cout << "  Strateji: ct_feat[j][slot_i] = ornek_i ozellik_j" << std::endl;
    std::cout << "  " << n_samples << " ornek PARALEL, 1 bootstrap" << std::endl;
    std::cout << "========================================" << std::endl;

    // ── ADIM 1: SIMD Batch Sifreleme ──────────────────────────────────────
    // ct_feat[j]: tum orneklerin j. ozelligi tek bir ciphertext'e
    // slot[i] = all_features[i][j]
    auto t_enc_start = Clock::now();

    std::vector<Ciphertext<DCRTPoly>> ct_feat(n_features);
    for (int j = 0; j < n_features; j++) {
        std::vector<std::complex<double>> slot_vec(numSlots, {0.0, 0.0});
        for (int i = 0; i < n_samples; i++) {
            slot_vec[i] = {all_features[i][j], 0.0};
        }
        Plaintext ptxt = cc->MakeCKKSPackedPlaintext(slot_vec, 1, 0, nullptr, numSlots);
        ct_feat[j] = cc->Encrypt(keyPair.publicKey, ptxt);
    }

    auto t_enc_end = Clock::now();
    double dt_enc = duration_cast<Ms>(t_enc_end - t_enc_start).count();
    std::cout << "\n[1] SIMD Batch Sifreleme   : " << std::fixed << std::setprecision(2)
              << dt_enc << " ms  (" << n_features << " ct, " << n_samples << " ornek paralel)" << std::endl;

    // ── ADIM 2: SIMD wx — her agirlik tum slot'lara yayilir ───────────────
    // w[j] tum slot'larda ayni deger → ct_feat[j] ile carpilinca
    // slot[i] = w[j] * ornek_i_ozellik_j
    auto t_mult_start = Clock::now();

    Ciphertext<DCRTPoly> ct_dot;
    bool first = true;
    for (int j = 0; j < n_features; j++) {
        // w[j] tum slot'lara yayilmis plaintext
        std::vector<std::complex<double>> w_vec(numSlots, {weights[j], 0.0});
        Plaintext ptxt_w = cc->MakeCKKSPackedPlaintext(w_vec, 1, 0, nullptr, numSlots);

        auto ct_wx_j = cc->EvalMult(ct_feat[j], ptxt_w);
        cc->RescaleInPlace(ct_wx_j);

        if (first) { ct_dot = ct_wx_j; first = false; }
        else       { ct_dot = cc->EvalAdd(ct_dot, ct_wx_j); }
    }
    // ct_dot[i] = sum_j(w[j] * x[i][j]) = w^T x[i]  tum ornekler icin

    auto t_mult_end = Clock::now();
    double dt_mult = duration_cast<Ms>(t_mult_end - t_mult_start).count();
    std::cout << "[2] SIMD EvalMult+Add      : " << std::fixed << std::setprecision(2)
              << dt_mult << " ms  (tum ornekler paralel)" << std::endl;

    // ── ADIM 3: Bias — tum slot'lara ayni deger ───────────────────────────
    // bias skaler olarak tum slot'lara eklenir
    auto t_bias_start = Clock::now();
    auto ct_logit = cc->EvalAdd(ct_dot, bias);
    auto t_bias_end = Clock::now();
    double dt_bias = duration_cast<Ms>(t_bias_end - t_bias_start).count();
    std::cout << "[3] Bias ekleme (SIMD)     : " << std::fixed << std::setprecision(2)
              << dt_bias << " ms  (Level: " << ct_logit->GetLevel() << ")" << std::endl;

    // ── ADIM 4: Level hizalama ─────────────────────────────────────────────
    auto t_align_start = Clock::now();
    while (ct_logit->GetLevel() < targetLevel)
        cc->LevelReduceInPlace(ct_logit, nullptr);
    auto t_align_end = Clock::now();
    double dt_align = duration_cast<Ms>(t_align_end - t_align_start).count();
    std::cout << "[4] Level hizalama         : " << std::fixed << std::setprecision(2)
              << dt_align << " ms  (Level: " << ct_logit->GetLevel() << "/" << targetLevel << ")" << std::endl;

    // ── ADIM 5: TEK Bootstrap — 30 ornek paralel ───────────────────────────
    // EvalFuncBootstrap ct_logit uzerinde calisir
    // Her slot[i] bagimsiz olarak f(logit_i) hesaplanir — SIMD
    auto t_bs_start = Clock::now();
    auto ct_result = cc->EvalFuncBootstrap(ct_logit, f_lut, p, 1);
    auto t_bs_end = Clock::now();
    double dt_bs = duration_cast<Ms>(t_bs_end - t_bs_start).count();
    std::cout << "[5] EvalFuncBootstrap(SIMD): " << std::fixed << std::setprecision(2)
              << dt_bs << " ms  (1 kez, " << n_samples << " ornek paralel)" << std::endl;

    // ── ADIM 6: Tek Decrypt — tum sonuclar ────────────────────────────────
    auto t_dec_start = Clock::now();
    Plaintext pt_result;
    cc->Decrypt(keyPair.secretKey, ct_result, &pt_result);
    pt_result->SetLength(n_samples);
    auto t_dec_end = Clock::now();
    double dt_dec = duration_cast<Ms>(t_dec_end - t_dec_start).count();
    std::cout << "[6] Sifre cozme (tek)      : " << std::fixed << std::setprecision(2)
              << dt_dec << " ms" << std::endl;

    // ── Sonuclari ayikla ──────────────────────────────────────────────────
    auto fhe_values = pt_result->GetCKKSPackedValue();

    std::cout << "\n========================================" << std::endl;
    std::cout << "  Sonuclar  (slot[i] = ornek i)" << std::endl;
    std::cout << "========================================" << std::endl;

    int correct = 0;
    double total_infer = dt_enc + dt_mult + dt_bias + dt_align + dt_bs + dt_dec;

    for (int i = 0; i < n_samples; i++) {
        double fhe_val = fhe_values[i].real();
        int fhe_pred   = (fhe_val > 0.5) ? 1 : 0;
        bool match     = (fhe_pred == pred_pt_all[i]);
        if (match) correct++;

        std::cout << "  Ornek " << (i+1) << ": "
                  << "logit=" << std::fixed << std::setprecision(4) << (bias + [&](){
                        double s = 0;
                        for (int j = 0; j < n_features; j++) s += weights[j]*all_features[i][j];
                        return s;
                     }())
                  << "  FHE=" << std::fixed << std::setprecision(6) << fhe_val
                  << "  pred=" << fhe_pred
                  << "  gercek=" << true_labels[i]
                  << (match ? "  [OK]" : "  [HATA]") << std::endl;
    }

    std::cout << "\n========================================" << std::endl;
    std::cout << "  HESAPLAMA PROFILI  (SIMD, " << n_samples << " ornek)" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << std::fixed << std::setprecision(2);
    std::cout << "  Anahtar kurulum (toplam) : " << t_setup   << " ms" << std::endl;
    std::cout << "    -- Key uretimi         : " << t_keygen  << " ms" << std::endl;
    std::cout << "    -- Bootstrap key       : " << t_bskey   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Sifreli cikarim (toplam) : " << total_infer << " ms" << std::endl;
    std::cout << "    -- SIMD Sifreleme      : " << dt_enc   << " ms  (" << n_features << " ct)" << std::endl;
    std::cout << "    -- SIMD EvalMult+Add   : " << dt_mult  << " ms" << std::endl;
    std::cout << "    -- Bias ekleme         : " << dt_bias  << " ms" << std::endl;
    std::cout << "    -- Level hizalama      : " << dt_align << " ms" << std::endl;
    std::cout << "    -- EvalFuncBootstrap   : " << dt_bs    << " ms  (1 kez, " << n_samples << " paralel)" << std::endl;
    std::cout << "    -- Sifre cozme         : " << dt_dec   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Ornek basi amortize      : " << (total_infer / n_samples) << " ms" << std::endl;
    std::cout << "  Bootstrap orani          : " << (100.0 * dt_bs / total_infer) << " %" << std::endl;
    std::cout << "  ================================" << std::endl;
    std::cout << "  SONUC: " << correct << "/" << n_samples
              << "  Dogruluk: " << (100.0 * correct / n_samples) << " %" << std::endl;
    std::cout << "========================================\n" << std::endl;
}

int main() {
    IrisFunctionalBootstrapping();
    return 0;
}
"""

with open("/content/sec-ckks/src/pke/examples/iris_functionallite.cpp", "w") as f:
    f.write(cpp_code)

print("Dosya yazildi.")

In [ ]:
cpp_code = r"""
#include "openfhe.h"
#include <complex>
#include <iostream>
#include <iomanip>
#include <vector>
#include <chrono>
using namespace lbcrypto;
using namespace std::chrono;

using Clock = high_resolution_clock;
using Ms    = duration<double, std::milli>;

void IrisFunctionalBootstrapping() {

    CCParams<CryptoContextCKKSRNS> parameters;
    parameters.SetSecretKeyDist(SPARSE_TERNARY);
    parameters.SetSecurityLevel(HEStd_128_classic);
    parameters.SetRingDim(1 << 13);
    parameters.SetNumLargeDigits(3);
    parameters.SetKeySwitchTechnique(HYBRID);
    parameters.SetScalingModSize(45);
    parameters.SetScalingTechnique(FIXEDMANUAL);
    parameters.SetFirstModSize(46);

    std::vector<uint32_t> levelBudget = {4, 2};
    std::vector<uint32_t> bsgsDim     = {0, 0};
    int bits = 5;
    int p    = 32;

    usint depth = levelBudget[0] + levelBudget[1] + 1 + 5 + 4 + bits + 2;
    parameters.SetMultiplicativeDepth(depth);

    std::cout << "\n========================================" << std::endl;
    std::cout << "  SIMD CKKS — Anahtar Kurulum" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << "  p (modul)           : " << p << "  (guvenli aralik: (-16, +16))" << std::endl;
    std::cout << "  bits                : " << bits << std::endl;
    std::cout << "  depth               : " << depth << std::endl;

    auto t_setup_start = Clock::now();

    CryptoContext<DCRTPoly> cc = GenCryptoContext(parameters);
    cc->Enable(PKE);
    cc->Enable(KEYSWITCH);
    cc->Enable(LEVELEDSHE);
    cc->Enable(ADVANCEDSHE);
    cc->Enable(FHE);
    cc->Enable(FBTS);

    int numSlots = cc->GetRingDimension() / 2;
    cc->EvalFuncBootstrapSetup(levelBudget, bsgsDim, numSlots, bits);

    auto t_keygen_start = Clock::now();
    auto keyPair = cc->KeyGen();
    cc->EvalMultKeyGen(keyPair.secretKey);
    auto t_keygen_end = Clock::now();

    auto t_bskey_start = Clock::now();
    cc->EvalBootstrapKeyGen(keyPair.secretKey, numSlots);
    auto t_bskey_end = Clock::now();

    // Rotasyon anahtarlari: iç çarpim toplamı için (önceki kodla aynı)
    cc->EvalRotateKeyGen(keyPair.secretKey, {1, 2});

    auto t_setup_end = Clock::now();

    double t_keygen = duration_cast<Ms>(t_keygen_end - t_keygen_start).count();
    double t_bskey  = duration_cast<Ms>(t_bskey_end  - t_bskey_start).count();
    double t_setup  = duration_cast<Ms>(t_setup_end  - t_setup_start).count();

    std::cout << "  Ring boyutu (n)     : " << cc->GetRingDimension() << std::endl;
    std::cout << "  Slot sayisi         : " << numSlots << std::endl;
    std::cout << "  Carpim derinligi    : " << depth << std::endl;
    std::cout << "  Key uretimi         : " << std::fixed << std::setprecision(2) << t_keygen << " ms" << std::endl;
    std::cout << "  Bootstrap key       : " << std::fixed << std::setprecision(2) << t_bskey  << " ms" << std::endl;
    std::cout << "  Toplam kurulum      : " << std::fixed << std::setprecision(2) << t_setup  << " ms" << std::endl;

    // ── Veri ──────────────────────────────────────────────────────────────
    // 30 ornek x 4 ozellik
    std::vector<std::vector<double>> all_features = {
        {  0.3110,  -0.5924,   0.5354,   0.0009},
        { -0.1737,   1.7096,  -1.1697,  -1.1838},
        {  2.2497,  -1.0528,   1.7858,   1.4488},
        {  0.1898,  -0.3622,   0.4217,   0.3958},
        {  1.1592,  -0.5924,   0.5922,   0.2641},
        { -0.5372,   0.7888,  -1.2834,  -1.0522},
        { -0.2948,  -0.3622,  -0.0898,   0.1325},
        {  1.2803,   0.0982,   0.7628,   1.4488},
        {  0.4322,  -1.9736,   0.4217,   0.3958},
        { -0.0525,  -0.8226,   0.0807,   0.0009},
        {  0.7957,   0.3284,   0.7628,   1.0539},
        { -1.2642,  -0.1320,  -1.3402,  -1.4471},
        { -0.4160,   1.0190,  -1.3971,  -1.3154},
        { -1.1430,   0.0982,  -1.2834,  -1.4471},
        { -0.9007,   1.7096,  -1.2834,  -1.1838},
        {  0.5533,   0.5586,   0.5354,   0.5274},
        {  0.7957,  -0.1320,   1.1606,   1.3172},
        { -0.2948,  -1.2830,   0.0807,  -0.1308},
        { -0.1737,  -0.5924,   0.4217,   0.1325},
        {  0.6745,  -0.5924,   1.0469,   1.3172},
        { -1.3854,   0.3284,  -1.2266,  -1.3154},
        {  0.3110,  -0.1320,   0.6491,   0.7907},
        { -1.0218,   0.7888,  -1.2266,  -1.0522},
        {  0.6745,  -0.5924,   1.0469,   1.1856},
        {  2.4920,   1.7096,   1.5016,   1.0539},
        {  1.0380,  -0.1320,   0.8196,   1.4488},
        {  1.0380,  -1.2830,   1.1606,   0.7907},
        {  1.1592,   0.3284,   1.2175,   1.4488},
        { -1.2642,  -0.1320,  -1.3402,  -1.1838},
        { -1.2642,   0.0982,  -1.2266,  -1.3154}
    };
    std::vector<int> true_labels = {0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1};
    std::vector<double> weights  = {-1.0183, 1.1725, -1.6753, -1.5359};
    double bias = -2.4196;

    int n_samples  = (int)all_features.size();
    int n_features = 4;

    // LUT fonksiyonu — p=32 icin esik 16
    auto f_lut = [](double x) -> double {
        return (x < 16.0) ? 1.0 : 0.0;
    };

    usint targetLevel = depth - levelBudget[1];

    // ── Plaintext doğrulama ───────────────────────────────────────────────
    std::cout << "\n  Plaintext logitler (dogrulama):" << std::endl;
    std::vector<int> pred_pt_all(n_samples);
    for (int i = 0; i < n_samples; i++) {
        double logit_pt = bias;
        for (int j = 0; j < n_features; j++) logit_pt += weights[j] * all_features[i][j];
        pred_pt_all[i] = (logit_pt > 0.0) ? 1 : 0;
        bool in_range = (logit_pt > -16.0 && logit_pt < 16.0);
        std::cout << "    Ornek " << (i+1) << ": logit=" << std::fixed << std::setprecision(4)
                  << logit_pt << (in_range ? "  [OK]" : "  [ARALIK DISI]")
                  << "  pred=" << pred_pt_all[i] << std::endl;
    }

    std::cout << "\n========================================" << std::endl;
    std::cout << "  SIMD Batch Sifreli Cikarim" << std::endl;
    std::cout << "  Strateji: ct_feat[j][slot_i] = ornek_i ozellik_j" << std::endl;
    std::cout << "  " << n_samples << " ornek PARALEL, 1 bootstrap" << std::endl;
    std::cout << "========================================" << std::endl;

    // ── ADIM 1: SIMD Batch Sifreleme ──────────────────────────────────────
    // ct_feat[j]: tum orneklerin j. ozelligi tek bir ciphertext'e
    // slot[i] = all_features[i][j]
    auto t_enc_start = Clock::now();

    std::vector<Ciphertext<DCRTPoly>> ct_feat(n_features);
    for (int j = 0; j < n_features; j++) {
        std::vector<std::complex<double>> slot_vec(numSlots, {0.0, 0.0});
        for (int i = 0; i < n_samples; i++) {
            slot_vec[i] = {all_features[i][j], 0.0};
        }
        Plaintext ptxt = cc->MakeCKKSPackedPlaintext(slot_vec, 1, 0, nullptr, numSlots);
        ct_feat[j] = cc->Encrypt(keyPair.publicKey, ptxt);
    }

    auto t_enc_end = Clock::now();
    double dt_enc = duration_cast<Ms>(t_enc_end - t_enc_start).count();
    std::cout << "\n[1] SIMD Batch Sifreleme   : " << std::fixed << std::setprecision(2)
              << dt_enc << " ms  (" << n_features << " ct, " << n_samples << " ornek paralel)" << std::endl;

    // ── ADIM 2: SIMD wx — her agirlik tum slot'lara yayilir ───────────────
    // w[j] tum slot'larda ayni deger → ct_feat[j] ile carpilinca
    // slot[i] = w[j] * ornek_i_ozellik_j
    auto t_mult_start = Clock::now();

    Ciphertext<DCRTPoly> ct_dot;
    bool first = true;
    for (int j = 0; j < n_features; j++) {
        // w[j] tum slot'lara yayilmis plaintext
        std::vector<std::complex<double>> w_vec(numSlots, {weights[j], 0.0});
        Plaintext ptxt_w = cc->MakeCKKSPackedPlaintext(w_vec, 1, 0, nullptr, numSlots);

        auto ct_wx_j = cc->EvalMult(ct_feat[j], ptxt_w);
        cc->RescaleInPlace(ct_wx_j);

        if (first) { ct_dot = ct_wx_j; first = false; }
        else       { ct_dot = cc->EvalAdd(ct_dot, ct_wx_j); }
    }
    // ct_dot[i] = sum_j(w[j] * x[i][j]) = w^T x[i]  tum ornekler icin

    auto t_mult_end = Clock::now();
    double dt_mult = duration_cast<Ms>(t_mult_end - t_mult_start).count();
    std::cout << "[2] SIMD EvalMult+Add      : " << std::fixed << std::setprecision(2)
              << dt_mult << " ms  (tum ornekler paralel)" << std::endl;

    // ── ADIM 3: Bias — tum slot'lara ayni deger ───────────────────────────
    // bias skaler olarak tum slot'lara eklenir
    auto t_bias_start = Clock::now();
    auto ct_logit = cc->EvalAdd(ct_dot, bias);
    auto t_bias_end = Clock::now();
    double dt_bias = duration_cast<Ms>(t_bias_end - t_bias_start).count();
    std::cout << "[3] Bias ekleme (SIMD)     : " << std::fixed << std::setprecision(2)
              << dt_bias << " ms  (Level: " << ct_logit->GetLevel() << ")" << std::endl;

    // ── ADIM 4: Level hizalama ─────────────────────────────────────────────
    auto t_align_start = Clock::now();
    while (ct_logit->GetLevel() < targetLevel)
        cc->LevelReduceInPlace(ct_logit, nullptr);
    auto t_align_end = Clock::now();
    double dt_align = duration_cast<Ms>(t_align_end - t_align_start).count();
    std::cout << "[4] Level hizalama         : " << std::fixed << std::setprecision(2)
              << dt_align << " ms  (Level: " << ct_logit->GetLevel() << "/" << targetLevel << ")" << std::endl;

    // ── ADIM 5: TEK Bootstrap — 30 ornek paralel ───────────────────────────
    // EvalFuncBootstrap ct_logit uzerinde calisir
    // Her slot[i] bagimsiz olarak f(logit_i) hesaplanir — SIMD
    auto t_bs_start = Clock::now();
    auto ct_result = cc->EvalFuncBootstrap(ct_logit, f_lut, p, 1);
    auto t_bs_end = Clock::now();
    double dt_bs = duration_cast<Ms>(t_bs_end - t_bs_start).count();
    std::cout << "[5] EvalFuncBootstrap(SIMD): " << std::fixed << std::setprecision(2)
              << dt_bs << " ms  (1 kez, " << n_samples << " ornek paralel)" << std::endl;

    // ── ADIM 6: Tek Decrypt — tum sonuclar ────────────────────────────────
    auto t_dec_start = Clock::now();
    Plaintext pt_result;
    cc->Decrypt(keyPair.secretKey, ct_result, &pt_result);
    pt_result->SetLength(n_samples);
    auto t_dec_end = Clock::now();
    double dt_dec = duration_cast<Ms>(t_dec_end - t_dec_start).count();
    std::cout << "[6] Sifre cozme (tek)      : " << std::fixed << std::setprecision(2)
              << dt_dec << " ms" << std::endl;

    // ── Sonuclari ayikla ──────────────────────────────────────────────────
    auto fhe_values = pt_result->GetCKKSPackedValue();

    std::cout << "\n========================================" << std::endl;
    std::cout << "  Sonuclar  (slot[i] = ornek i)" << std::endl;
    std::cout << "========================================" << std::endl;

    int correct = 0;
    double total_infer = dt_enc + dt_mult + dt_bias + dt_align + dt_bs + dt_dec;

    for (int i = 0; i < n_samples; i++) {
        double fhe_val = fhe_values[i].real();
        int fhe_pred   = (fhe_val > 0.5) ? 1 : 0;
        bool match     = (fhe_pred == pred_pt_all[i]);
        if (match) correct++;

        std::cout << "  Ornek " << (i+1) << ": "
                  << "logit=" << std::fixed << std::setprecision(4) << (bias + [&](){
                        double s = 0;
                        for (int j = 0; j < n_features; j++) s += weights[j]*all_features[i][j];
                        return s;
                     }())
                  << "  FHE=" << std::fixed << std::setprecision(6) << fhe_val
                  << "  pred=" << fhe_pred
                  << "  gercek=" << true_labels[i]
                  << (match ? "  [OK]" : "  [HATA]") << std::endl;
    }

    std::cout << "\n========================================" << std::endl;
    std::cout << "  HESAPLAMA PROFILI  (SIMD, " << n_samples << " ornek)" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << std::fixed << std::setprecision(2);
    std::cout << "  Anahtar kurulum (toplam) : " << t_setup   << " ms" << std::endl;
    std::cout << "    -- Key uretimi         : " << t_keygen  << " ms" << std::endl;
    std::cout << "    -- Bootstrap key       : " << t_bskey   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Sifreli cikarim (toplam) : " << total_infer << " ms" << std::endl;
    std::cout << "    -- SIMD Sifreleme      : " << dt_enc   << " ms  (" << n_features << " ct)" << std::endl;
    std::cout << "    -- SIMD EvalMult+Add   : " << dt_mult  << " ms" << std::endl;
    std::cout << "    -- Bias ekleme         : " << dt_bias  << " ms" << std::endl;
    std::cout << "    -- Level hizalama      : " << dt_align << " ms" << std::endl;
    std::cout << "    -- EvalFuncBootstrap   : " << dt_bs    << " ms  (1 kez, " << n_samples << " paralel)" << std::endl;
    std::cout << "    -- Sifre cozme         : " << dt_dec   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Ornek basi amortize      : " << (total_infer / n_samples) << " ms" << std::endl;
    std::cout << "  Bootstrap orani          : " << (100.0 * dt_bs / total_infer) << " %" << std::endl;
    std::cout << "  ================================" << std::endl;
    std::cout << "  SONUC: " << correct << "/" << n_samples
              << "  Dogruluk: " << (100.0 * correct / n_samples) << " %" << std::endl;
    std::cout << "========================================\n" << std::endl;
}

int main() {
    IrisFunctionalBootstrapping();
    return 0;
}
"""

with open("/content/sec-ckks/src/pke/examples/iris_functionalite.cpp", "w") as f:
    f.write(cpp_code)

print("Dosya yazildi.")

**Output**

In [ ]:
/content/sec-ckks
[ 96%] Built target pkeobj
[ 96%] Built target OPENFHEpke
[ 96%] Building CXX object src/pke/CMakeFiles/iris_functional.dir/examples/iris_functional.cpp.o
[100%] Linking CXX executable ../../bin/examples/pke/iris_functional
[100%] Built target iris_functional

========================================
  Anahtar Kurulum
========================================
  p (modul)           : 32  (guvenli aralik: (-16, +16))
  bits                : 5
  depth               : 23
  Ring boyutu (n)     : 8192
  Slot sayisi         : 4096
  Carpim derinligi    : 23
  Key uretimi         : 136.55 ms
  Bootstrap key       : 1601.04 ms
  Toplam kurulum      : 2496.27 ms

========================================
  Sifreli Cikarim  (5 ornek)
========================================

--- Ornek 1/5 ---
  Ozellikler     : [0.311, -0.592, 0.535, 0.001]
  Gercek sinif   : 0
  Plaintext logit: -4.3283  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 23.56 ms
  [2] EvalMult+Rescale : 9.35 ms
  [3] Rotasyon toplami : 67.87 ms
  [4] Bias ekleme      : 0.52 ms  (Level: 1)
  [5] Level hizalama   : 0.13 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 2535.70 ms
  [7] Sifre cozme      : 5.17 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 2642.50 ms

--- Ornek 2/5 ---
  Ozellikler     : [-0.174, 1.710, -1.170, -1.184]
  Gercek sinif   : 1
  Plaintext logit: 3.5417  [aralik OK]
  Plaintext pred : 1
  [1] Sifreleme        : 24.58 ms
  [2] EvalMult+Rescale : 8.54 ms
  [3] Rotasyon toplami : 52.28 ms
  [4] Bias ekleme      : 0.49 ms  (Level: 1)
  [5] Level hizalama   : 0.09 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 2378.83 ms
  [7] Sifre cozme      : 5.08 ms
  --------------------------------
  FHE ham deger   : 1.000000
  FHE tahmin      : 1  [OK]
  Ornek suresi    : 2470.05 ms

--- Ornek 3/5 ---
  Ozellikler     : [2.250, -1.053, 1.786, 1.449]
  Gercek sinif   : 0
  Plaintext logit: -11.1625  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 24.49 ms
  [2] EvalMult+Rescale : 8.34 ms
  [3] Rotasyon toplami : 53.03 ms
  [4] Bias ekleme      : 0.49 ms  (Level: 1)
  [5] Level hizalama   : 0.09 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 2412.48 ms
  [7] Sifre cozme      : 4.92 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 2504.00 ms

--- Ornek 4/5 ---
  Ozellikler     : [0.190, -0.362, 0.422, 0.396]
  Gercek sinif   : 0
  Plaintext logit: -4.3528  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 24.88 ms
  [2] EvalMult+Rescale : 8.59 ms
  [3] Rotasyon toplami : 53.79 ms
  [4] Bias ekleme      : 0.49 ms  (Level: 1)
  [5] Level hizalama   : 0.09 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 3801.00 ms
  [7] Sifre cozme      : 5.14 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 3894.17 ms

--- Ornek 5/5 ---
  Ozellikler     : [1.159, -0.592, 0.592, 0.264]
  Gercek sinif   : 0
  Plaintext logit: -5.6910  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 24.66 ms
  [2] EvalMult+Rescale : 8.47 ms
  [3] Rotasyon toplami : 72.73 ms
  [4] Bias ekleme      : 5.99 ms  (Level: 1)
  [5] Level hizalama   : 0.10 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 2424.53 ms
  [7] Sifre cozme      : 4.78 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 2541.43 ms

========================================
  HESAPLAMA PROFILI  (5 ornek toplam)
========================================
  Anahtar kurulum (toplam) : 2496.27 ms
    -- Key uretimi         : 136.55 ms
    -- Bootstrap key       : 1601.04 ms
  --------------------------------
  Sifreli cikarim (toplam) : 14052.15 ms
    -- Sifreleme           : 122.17 ms
    -- EvalMult + Rescale  : 43.29 ms
    -- Rotasyon toplami    : 299.69 ms
    -- Bias ekleme         : 7.97 ms
    -- Level hizalama      : 0.49 ms
    -- EvalFuncBootstrap   : 13552.54 ms
    -- Sifre cozme         : 25.08 ms
  --------------------------------
  Ornek basi ortalama      : 2810.43 ms
  Bootstrap orani          : 96.44 %
  ================================
  SONUC: 5/5  Dogruluk: 100.00 %
========================================



In [ ]:
-- demoData folder already exists
[  0%] Built target third-party
[ 35%] Built target coreobj
[ 39%] Built target OPENFHEcore
[ 46%] Built target binfheobj
[ 46%] Built target OPENFHEbinfhe
[100%] Built target pkeobj
[100%] Built target OPENFHEpke
[100%] Building CXX object src/pke/CMakeFiles/iris_functionalite.dir/examples/iris_functionalite.cpp.o
[100%] Linking CXX executable ../../bin/examples/pke/iris_functionalite
[100%] Built target iris_functionalite

========================================
  Anahtar Kurulum
========================================
  p (modul)           : 32  (guvenli aralik: (-16, +16))
  bits                : 5
  depth               : 23
  Ring boyutu (n)     : 65536
  Slot sayisi         : 32768
  Carpim derinligi    : 23
  Key uretimi         : 1111.93 ms
  Bootstrap key       : 22167.98 ms
  Toplam kurulum      : 36630.67 ms

========================================
  Sifreli Cikarim  (30 ornek)
========================================

--- Ornek 1/30 ---
  Ozellikler     : [0.311, -0.592, 0.535, 0.001]
  Gercek sinif   : 0
  Plaintext logit: -4.3283  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 211.23 ms
  [2] EvalMult+Rescale : 86.38 ms
  [3] Rotasyon toplami : 631.15 ms
  [4] Bias ekleme      : 5.19 ms  (Level: 1)
  [5] Level hizalama   : 0.14 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 29271.62 ms
  [7] Sifre cozme      : 37.13 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 30243.13 ms

--- Ornek 2/30 ---
  Ozellikler     : [-0.174, 1.710, -1.170, -1.184]
  Gercek sinif   : 1
  Plaintext logit: 3.5417  [aralik OK]
  Plaintext pred : 1
  [1] Sifreleme        : 251.15 ms
  [2] EvalMult+Rescale : 106.33 ms
  [3] Rotasyon toplami : 741.51 ms
  [4] Bias ekleme      : 8.45 ms  (Level: 1)
  [5] Level hizalama   : 0.13 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28969.15 ms
  [7] Sifre cozme      : 38.49 ms
  --------------------------------
  FHE ham deger   : 1.000000
  FHE tahmin      : 1  [OK]
  Ornek suresi    : 30115.44 ms

--- Ornek 3/30 ---
  Ozellikler     : [2.250, -1.053, 1.786, 1.449]
  Gercek sinif   : 0
  Plaintext logit: -11.1625  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 230.24 ms
  [2] EvalMult+Rescale : 95.12 ms
  [3] Rotasyon toplami : 611.03 ms
  [4] Bias ekleme      : 5.12 ms  (Level: 1)
  [5] Level hizalama   : 0.11 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28621.24 ms
  [7] Sifre cozme      : 37.99 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 29601.11 ms

--- Ornek 4/30 ---
  Ozellikler     : [0.190, -0.362, 0.422, 0.396]
  Gercek sinif   : 0
  Plaintext logit: -4.3528  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 251.06 ms
  [2] EvalMult+Rescale : 101.94 ms
  [3] Rotasyon toplami : 725.40 ms
  [4] Bias ekleme      : 7.82 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28925.55 ms
  [7] Sifre cozme      : 35.89 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 30048.00 ms

--- Ornek 5/30 ---
  Ozellikler     : [1.159, -0.592, 0.592, 0.264]
  Gercek sinif   : 0
  Plaintext logit: -5.6910  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 210.24 ms
  [2] EvalMult+Rescale : 78.75 ms
  [3] Rotasyon toplami : 621.63 ms
  [4] Bias ekleme      : 4.98 ms  (Level: 1)
  [5] Level hizalama   : 0.11 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28293.06 ms
  [7] Sifre cozme      : 36.59 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 29245.63 ms

--- Ornek 6/30 ---
  Ozellikler     : [-0.901, -0.592, -1.284, -1.315]
  Gercek sinif   : 0
  Plaintext logit: 1.9737  [aralik OK]
  Plaintext pred : 1
  [1] Sifreleme        : 249.13 ms
  [2] EvalMult+Rescale : 104.39 ms
  [3] Rotasyon toplami : 720.01 ms
  [4] Bias ekleme      : 5.17 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 29097.92 ms
  [7] Sifre cozme      : 38.01 ms
  --------------------------------
  FHE ham deger   : 1.000000
  FHE tahmin      : 1  [OK]
  Ornek suresi    : 30214.95 ms

--- Ornek 7/30 ---
  Ozellikler     : [-1.143, -0.131, -1.341, -1.315]
  Gercek sinif   : 0
  Plaintext logit: 2.8563  [aralik OK]
  Plaintext pred : 1
  [1] Sifreleme        : 209.94 ms
  [2] EvalMult+Rescale : 77.57 ms
  [3] Rotasyon toplami : 607.68 ms
  [4] Bias ekleme      : 4.85 ms  (Level: 1)
  [5] Level hizalama   : 0.11 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28423.75 ms
  [7] Sifre cozme      : 39.62 ms
  --------------------------------
  FHE ham deger   : 1.000000
  FHE tahmin      : 1  [OK]
  Ornek suresi    : 29363.77 ms

--- Ornek 8/30 ---
  Ozellikler     : [-0.174, -0.131, -1.227, -1.315]
  Gercek sinif   : 0
  Plaintext logit: 1.6789  [aralik OK]
  Plaintext pred : 1
  [1] Sifreleme        : 249.22 ms
  [2] EvalMult+Rescale : 104.78 ms
  [3] Rotasyon toplami : 610.49 ms
  [4] Bias ekleme      : 5.00 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28479.44 ms
  [7] Sifre cozme      : 37.16 ms
  --------------------------------
  FHE ham deger   : 1.000000
  FHE tahmin      : 1  [OK]
  Ornek suresi    : 29486.44 ms

--- Ornek 9/30 ---
  Ozellikler     : [0.553, -1.743, 0.649, 0.396]
  Gercek sinif   : 0
  Plaintext logit: -6.7224  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 211.31 ms
  [2] EvalMult+Rescale : 77.80 ms
  [3] Rotasyon toplami : 617.48 ms
  [4] Bias ekleme      : 5.05 ms  (Level: 1)
  [5] Level hizalama   : 0.11 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28725.43 ms
  [7] Sifre cozme      : 39.39 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 29676.82 ms

--- Ornek 10/30 ---
  Ozellikler     : [-0.537, -0.822, 0.138, 0.132]
  Gercek sinif   : 0
  Plaintext logit: -3.2710  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 216.64 ms
  [2] EvalMult+Rescale : 78.12 ms
  [3] Rotasyon toplami : 628.55 ms
  [4] Bias ekleme      : 5.13 ms  (Level: 1)
  [5] Level hizalama   : 0.11 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 29116.36 ms
  [7] Sifre cozme      : 36.74 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 30081.89 ms

--- Ornek 11/30 ---
  Ozellikler     : [0.795, 0.329, 0.763, 1.054]
  Gercek sinif   : 1
  Plaintext logit: -5.7400  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 211.36 ms
  [2] EvalMult+Rescale : 78.37 ms
  [3] Rotasyon toplami : 627.03 ms
  [4] Bias ekleme      : 5.33 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 29190.43 ms
  [7] Sifre cozme      : 39.58 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 30152.48 ms

--- Ornek 12/30 ---
  Ozellikler     : [-1.264, -0.131, -1.341, -1.447]
  Gercek sinif   : 0
  Plaintext logit: 3.1823  [aralik OK]
  Plaintext pred : 1
  [1] Sifreleme        : 210.87 ms
  [2] EvalMult+Rescale : 78.67 ms
  [3] Rotasyon toplami : 625.63 ms
  [4] Bias ekleme      : 5.29 ms  (Level: 1)
  [5] Level hizalama   : 0.11 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 29372.63 ms
  [7] Sifre cozme      : 38.66 ms
  --------------------------------
  FHE ham deger   : 1.000000
  FHE tahmin      : 1  [OK]
  Ornek suresi    : 30332.10 ms

--- Ornek 13/30 ---
  Ozellikler     : [-0.295, -0.592, -0.138, 0.001]
  Gercek sinif   : 0
  Plaintext logit: -2.5841  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 213.69 ms
  [2] EvalMult+Rescale : 79.21 ms
  [3] Rotasyon toplami : 622.21 ms
  [4] Bias ekleme      : 5.32 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28387.57 ms
  [7] Sifre cozme      : 36.89 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 29345.29 ms

--- Ornek 14/30 ---
  Ozellikler     : [0.069, -1.283, 0.706, 0.659]
  Gercek sinif   : 0
  Plaintext logit: -6.1896  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 211.32 ms
  [2] EvalMult+Rescale : 94.79 ms
  [3] Rotasyon toplami : 621.36 ms
  [4] Bias ekleme      : 5.14 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 29400.13 ms
  [7] Sifre cozme      : 39.29 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 30372.37 ms

--- Ornek 15/30 ---
  Ozellikler     : [-0.416, 1.020, -1.398, -1.315]
  Gercek sinif   : 0
  Plaintext logit: 3.5618  [aralik OK]
  Plaintext pred : 1
  [1] Sifreleme        : 210.86 ms
  [2] EvalMult+Rescale : 78.85 ms
  [3] Rotasyon toplami : 625.62 ms
  [4] Bias ekleme      : 5.30 ms  (Level: 1)
  [5] Level hizalama   : 0.11 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 29270.46 ms
  [7] Sifre cozme      : 36.40 ms
  --------------------------------
  FHE ham deger   : 1.000000
  FHE tahmin      : 1  [OK]
  Ornek suresi    : 30227.85 ms

--- Ornek 16/30 ---
  Ozellikler     : [1.280, 0.099, 0.592, 0.264]
  Gercek sinif   : 0
  Plaintext logit: -5.0036  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 216.02 ms
  [2] EvalMult+Rescale : 78.90 ms
  [3] Rotasyon toplami : 622.55 ms
  [4] Bias ekleme      : 5.34 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 29064.90 ms
  [7] Sifre cozme      : 36.47 ms
  --------------------------------
  FHE ham deger   : -0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 30024.52 ms

--- Ornek 17/30 ---
  Ozellikler     : [0.432, 0.789, 0.933, 1.449]
  Gercek sinif   : 1
  Plaintext logit: -5.7223  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 247.98 ms
  [2] EvalMult+Rescale : 108.27 ms
  [3] Rotasyon toplami : 745.20 ms
  [4] Bias ekleme      : 7.85 ms  (Level: 1)
  [5] Level hizalama   : 0.13 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 29183.80 ms
  [7] Sifre cozme      : 36.16 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 30329.62 ms

--- Ornek 18/30 ---
  Ozellikler     : [-1.022, 0.099, -1.284, -1.315]
  Gercek sinif   : 0
  Plaintext logit: 2.9075  [aralik OK]
  Plaintext pred : 1
  [1] Sifreleme        : 210.92 ms
  [2] EvalMult+Rescale : 71.85 ms
  [3] Rotasyon toplami : 602.21 ms
  [4] Bias ekleme      : 5.24 ms  (Level: 1)
  [5] Level hizalama   : 0.13 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28014.97 ms
  [7] Sifre cozme      : 36.73 ms
  --------------------------------
  FHE ham deger   : 1.000000
  FHE tahmin      : 1  [OK]
  Ornek suresi    : 28942.27 ms

--- Ornek 19/30 ---
  Ozellikler     : [-0.537, -1.283, 0.366, 0.001]
  Gercek sinif   : 0
  Plaintext logit: -3.9925  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 257.39 ms
  [2] EvalMult+Rescale : 105.69 ms
  [3] Rotasyon toplami : 731.63 ms
  [4] Bias ekleme      : 7.82 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 29263.88 ms
  [7] Sifre cozme      : 36.53 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 30403.33 ms

--- Ornek 20/30 ---
  Ozellikler     : [1.038, -0.131, 0.820, 1.449]
  Gercek sinif   : 1
  Plaintext logit: -7.2291  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 227.70 ms
  [2] EvalMult+Rescale : 77.97 ms
  [3] Rotasyon toplami : 617.01 ms
  [4] Bias ekleme      : 5.18 ms  (Level: 1)
  [5] Level hizalama   : 0.13 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28342.60 ms
  [7] Sifre cozme      : 38.61 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 29309.43 ms

--- Ornek 21/30 ---
  Ozellikler     : [-0.780, -0.822, -0.252, -0.264]
  Gercek sinif   : 0
  Plaintext logit: -1.7622  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 245.99 ms
  [2] EvalMult+Rescale : 101.75 ms
  [3] Rotasyon toplami : 749.81 ms
  [4] Bias ekleme      : 7.72 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28859.11 ms
  [7] Sifre cozme      : 38.57 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 30003.31 ms

--- Ornek 22/30 ---
  Ozellikler     : [0.674, -0.592, 1.047, 1.317]
  Gercek sinif   : 1
  Plaintext logit: -7.5768  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 210.34 ms
  [2] EvalMult+Rescale : 78.14 ms
  [3] Rotasyon toplami : 629.89 ms
  [4] Bias ekleme      : 5.23 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28082.10 ms
  [7] Sifre cozme      : 36.69 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 29042.76 ms

--- Ornek 23/30 ---
  Ozellikler     : [-1.022, 0.329, -1.455, -1.315]
  Gercek sinif   : 0
  Plaintext logit: 3.4637  [aralik OK]
  Plaintext pred : 1
  [1] Sifreleme        : 257.38 ms
  [2] EvalMult+Rescale : 107.37 ms
  [3] Rotasyon toplami : 756.13 ms
  [4] Bias ekleme      : 7.81 ms  (Level: 1)
  [5] Level hizalama   : 0.13 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 29238.02 ms
  [7] Sifre cozme      : 39.39 ms
  --------------------------------
  FHE ham deger   : 1.000000
  FHE tahmin      : 1  [OK]
  Ornek suresi    : 30406.50 ms

--- Ornek 24/30 ---
  Ozellikler     : [0.311, -0.131, 0.478, 0.264]
  Gercek sinif   : 0
  Plaintext logit: -4.0960  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 211.53 ms
  [2] EvalMult+Rescale : 101.86 ms
  [3] Rotasyon toplami : 629.13 ms
  [4] Bias ekleme      : 8.84 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28216.14 ms
  [7] Sifre cozme      : 39.04 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 29206.88 ms

--- Ornek 25/30 ---
  Ozellikler     : [-0.901, -1.283, -0.423, -0.132]
  Gercek sinif   : 0
  Plaintext logit: -2.0961  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 248.48 ms
  [2] EvalMult+Rescale : 104.01 ms
  [3] Rotasyon toplami : 722.18 ms
  [4] Bias ekleme      : 5.49 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28754.95 ms
  [7] Sifre cozme      : 40.14 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 29875.62 ms

--- Ornek 26/30 ---
  Ozellikler     : [1.522, 1.249, 1.332, 1.712]
  Gercek sinif   : 1
  Plaintext logit: -7.3647  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 210.77 ms
  [2] EvalMult+Rescale : 79.00 ms
  [3] Rotasyon toplami : 631.62 ms
  [4] Bias ekleme      : 5.19 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28285.60 ms
  [7] Sifre cozme      : 48.82 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 29261.35 ms

--- Ornek 27/30 ---
  Ozellikler     : [0.553, 0.789, 1.047, 1.580]
  Gercek sinif   : 1
  Plaintext logit: -6.2377  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 250.77 ms
  [2] EvalMult+Rescale : 102.09 ms
  [3] Rotasyon toplami : 729.96 ms
  [4] Bias ekleme      : 5.16 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28750.85 ms
  [7] Sifre cozme      : 36.89 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 29876.09 ms

--- Ornek 28/30 ---
  Ozellikler     : [-0.295, -0.362, 0.138, 0.132]
  Gercek sinif   : 0
  Plaintext logit: -2.9778  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 242.61 ms
  [2] EvalMult+Rescale : 78.46 ms
  [3] Rotasyon toplami : 630.04 ms
  [4] Bias ekleme      : 5.33 ms  (Level: 1)
  [5] Level hizalama   : 0.12 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28031.34 ms
  [7] Sifre cozme      : 36.76 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 29024.91 ms

--- Ornek 29/30 ---
  Ozellikler     : [-1.264, -0.131, -1.341, -1.315]
  Gercek sinif   : 0
  Plaintext logit: 2.9795  [aralik OK]
  Plaintext pred : 1
  [1] Sifreleme        : 254.58 ms
  [2] EvalMult+Rescale : 104.37 ms
  [3] Rotasyon toplami : 740.14 ms
  [4] Bias ekleme      : 8.83 ms  (Level: 1)
  [5] Level hizalama   : 0.13 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 29047.34 ms
  [7] Sifre cozme      : 38.35 ms
  --------------------------------
  FHE ham deger   : 1.000000
  FHE tahmin      : 1  [OK]
  Ornek suresi    : 30194.00 ms

--- Ornek 30/30 ---
  Ozellikler     : [0.432, -0.592, 0.592, 0.790]
  Gercek sinif   : 0
  Plaintext logit: -5.7588  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 211.19 ms
  [2] EvalMult+Rescale : 78.77 ms
  [3] Rotasyon toplami : 614.82 ms
  [4] Bias ekleme      : 5.17 ms  (Level: 1)
  [5] Level hizalama   : 0.11 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 28180.48 ms
  [7] Sifre cozme      : 44.72 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 29135.50 ms

========================================
  HESAPLAMA PROFILI  (30 ornek toplam)
========================================
  Anahtar kurulum (toplam) : 36630.67 ms
    -- Key uretimi         : 1111.93 ms
    -- Bootstrap key       : 22167.98 ms
  --------------------------------
  Sifreli cikarim (toplam) : 893543.36 ms
    -- Sifreleme           : 6851.93 ms
    -- EvalMult + Rescale  : 2699.56 ms
    -- Rotasyon toplami    : 19789.11 ms
    -- Bias ekleme         : 179.34 ms
    -- Level hizalama      : 3.59 ms
    -- EvalFuncBootstrap   : 862860.82 ms
    -- Sifre cozme         : 1151.69 ms
  --------------------------------
  Ornek basi ortalama      : 29784.78 ms
  Bootstrap orani          : 96.57 %
  ================================
  SONUC: 30/30  Dogruluk: 100.00 %
========================================



In [ ]:
--- Content of iris_functionallite.cpp before compilation ---


#include "openfhe.h"
#include <complex>
#include <iostream>
#include <iomanip>
#include <vector>
#include <chrono>
using namespace lbcrypto;
using namespace std::chrono;

using Clock = high_resolution_clock;
using Ms    = duration<double, std::milli>;

void IrisFunctionalBootstrapping() {

    CCParams<CryptoContextCKKSRNS> parameters;
    parameters.SetSecretKeyDist(SPARSE_TERNARY);
    parameters.SetSecurityLevel(HEStd_128_classic);
    parameters.SetRingDim(1 << 16);
    parameters.SetNumLargeDigits(3);
    parameters.SetKeySwitchTechnique(HYBRID);
    parameters.SetScalingModSize(45);
    parameters.SetScalingTechnique(FIXEDMANUAL);
    parameters.SetFirstModSize(46);

    std::vector<uint32_t> levelBudget = {4, 2};
    std::vector<uint32_t> bsgsDim     = {0, 0};
    int bits = 5;
    int p    = 32;

    usint depth = levelBudget[0] + levelBudget[1] + 1 + 5 + 4 + bits + 2;
    parameters.SetMultiplicativeDepth(depth);

    std::cout << "\n========================================" << std::endl;
    std::cout << "  SIMD CKKS — Anahtar Kurulum" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << "  p (modul)           : " << p << "  (guvenli aralik: (-16, +16))" << std::endl;
    std::cout << "  bits                : " << bits << std::endl;
    std::cout << "  depth               : " << depth << std::endl;

    auto t_setup_start = Clock::now();

    CryptoContext<DCRTPoly> cc = GenCryptoContext(parameters);
    cc->Enable(PKE);
    cc->Enable(KEYSWITCH);
    cc->Enable(LEVELEDSHE);
    cc->Enable(ADVANCEDSHE);
    cc->Enable(FHE);
    cc->Enable(FBTS);

    int numSlots = cc->GetRingDimension() / 2;
    cc->EvalFuncBootstrapSetup(levelBudget, bsgsDim, numSlots, bits);

    auto t_keygen_start = Clock::now();
    auto keyPair = cc->KeyGen();
    cc->EvalMultKeyGen(keyPair.secretKey);
    auto t_keygen_end = Clock::now();

    auto t_bskey_start = Clock::now();
    cc->EvalBootstrapKeyGen(keyPair.secretKey, numSlots);
    auto t_bskey_end = Clock::now();

    // Rotasyon anahtarlari: iç çarpim toplami için (önceki kodla ayni)
    cc->EvalRotateKeyGen(keyPair.secretKey, {1, 2});

    auto t_setup_end = Clock::now();

    double t_keygen = duration_cast<Ms>(t_keygen_end - t_keygen_start).count();
    double t_bskey  = duration_cast<Ms>(t_bskey_end  - t_bskey_start).count();
    double t_setup  = duration_cast<Ms>(t_setup_end  - t_setup_start).count();

    std::cout << "  Ring boyutu (n)     : " << cc->GetRingDimension() << std::endl;
    std::cout << "  Slot sayisi         : " << numSlots << std::endl;
    std::cout << "  Carpim derinligi    : " << depth << std::endl;
    std::cout << "  Key uretimi         : " << std::fixed << std::setprecision(2) << t_keygen << " ms" << std::endl;
    std::cout << "  Bootstrap key       : " << std::fixed << std::setprecision(2) << t_bskey  << " ms" << std::endl;
    std::cout << "  Toplam kurulum      : " << std::fixed << std::setprecision(2) << t_setup  << " ms" << std::endl;

    // ── Veri ──────────────────────────────────────────────────────────────
    // 30 ornek x 4 ozellik
    std::vector<std::vector<double>> all_features = {
        {  0.3110,  -0.5924,   0.5354,   0.0009},
        { -0.1737,   1.7096,  -1.1697,  -1.1838},
        {  2.2497,  -1.0528,   1.7858,   1.4488},
        {  0.1898,  -0.3622,   0.4217,   0.3958},
        {  1.1592,  -0.5924,   0.5922,   0.2641},
        { -0.5372,   0.7888,  -1.2834,  -1.0522},
        { -0.2948,  -0.3622,  -0.0898,   0.1325},
        {  1.2803,   0.0982,   0.7628,   1.4488},
        {  0.4322,  -1.9736,   0.4217,   0.3958},
        { -0.0525,  -0.8226,   0.0807,   0.0009},
        {  0.7957,   0.3284,   0.7628,   1.0539},
        { -1.2642,  -0.1320,  -1.3402,  -1.4471},
        { -0.4160,   1.0190,  -1.3971,  -1.3154},
        { -1.1430,   0.0982,  -1.2834,  -1.4471},
        { -0.9007,   1.7096,  -1.2834,  -1.1838},
        {  0.5533,   0.5586,   0.5354,   0.5274},
        {  0.7957,  -0.1320,   1.1606,   1.3172},
        { -0.2948,  -1.2830,   0.0807,  -0.1308},
        { -0.1737,  -0.5924,   0.4217,   0.1325},
        {  0.6745,  -0.5924,   1.0469,   1.3172},
        { -1.3854,   0.3284,  -1.2266,  -1.3154},
        {  0.3110,  -0.1320,   0.6491,   0.7907},
        { -1.0218,   0.7888,  -1.2266,  -1.0522},
        {  0.6745,  -0.5924,   1.0469,   1.1856},
        {  2.4920,   1.7096,   1.5016,   1.0539},
        {  1.0380,  -0.1320,   0.8196,   1.4488},
        {  1.0380,  -1.2830,   1.1606,   0.7907},
        {  1.1592,   0.3284,   1.2175,   1.4488},
        { -1.2642,  -0.1320,  -1.3402,  -1.1838},
        { -1.2642,   0.0982,  -1.2266,  -1.3154}
    };
    std::vector<int> true_labels = {0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1};
    std::vector<double> weights  = {-1.0183, 1.1725, -1.6753, -1.5359};
    double bias = -2.4196;

    int n_samples  = (int)all_features.size();
    int n_features = 4;

    // LUT fonksiyonu — p=32 icin esik 16
    auto f_lut = [](double x) -> double {
        return (x < 16.0) ? 1.0 : 0.0;
    };

    usint targetLevel = depth - levelBudget[1];

    // ── Plaintext dogrulama ───────────────────────────────────────────────
    std::cout << "\n  Plaintext logitler (dogrulama):" << std::endl;
    std::vector<int> pred_pt_all(n_samples);
    for (int i = 0; i < n_samples; i++) {
        double logit_pt = bias;
        for (int j = 0; j < n_features; j++) logit_pt += weights[j] * all_features[i][j];
        pred_pt_all[i] = (logit_pt > 0.0) ? 1 : 0;
        bool in_range = (logit_pt > -16.0 && logit_pt < 16.0);
        std::cout << "    Ornek " << (i+1) << ": logit=" << std::fixed << std::setprecision(4)
                  << logit_pt << (in_range ? "  [OK]" : "  [ARALIK DISI]")
                  << "  pred=" << pred_pt_all[i] << std::endl;
    }

    std::cout << "\n========================================" << std::endl;
    std::cout << "  SIMD Batch Sifreli Cikarim" << std::endl;
    std::cout << "  Strateji: ct_feat[j][slot_i] = ornek_i ozellik_j" << std::endl;
    std::cout << "  " << n_samples << " ornek PARALEL, 1 bootstrap" << std::endl;
    std::cout << "========================================" << std::endl;

    // ── ADIM 1: SIMD Batch Sifreleme ──────────────────────────────────────
    // ct_feat[j]: tum orneklerin j. ozelligi tek bir ciphertext'e
    // slot[i] = all_features[i][j]
    auto t_enc_start = Clock::now();

    std::vector<Ciphertext<DCRTPoly>> ct_feat(n_features);
    for (int j = 0; j < n_features; j++) {
        std::vector<std::complex<double>> slot_vec(numSlots, {0.0, 0.0});
        for (int i = 0; i < n_samples; i++) {
            slot_vec[i] = {all_features[i][j], 0.0};
        }
        Plaintext ptxt = cc->MakeCKKSPackedPlaintext(slot_vec, 1, 0, nullptr, numSlots);
        ct_feat[j] = cc->Encrypt(keyPair.publicKey, ptxt);
    }

    auto t_enc_end = Clock::now();
    double dt_enc = duration_cast<Ms>(t_enc_end - t_enc_start).count();
    std::cout << "\n[1] SIMD Batch Sifreleme   : " << std::fixed << std::setprecision(2)
              << dt_enc << " ms  (" << n_features << " ct, " << n_samples << " ornek paralel)" << std::endl;

    // ── ADIM 2: SIMD wx — her agirlik tum slot'lara yayilir ───────────────
    // w[j] tum slot'larda ayni deger → ct_feat[j] ile carpilinca
    // slot[i] = w[j] * ornek_i_ozellik_j
    auto t_mult_start = Clock::now();

    Ciphertext<DCRTPoly> ct_dot;
    bool first = true;
    for (int j = 0; j < n_features; j++) {
        // w[j] tum slot'lara yayilmis plaintext
        std::vector<std::complex<double>> w_vec(numSlots, {weights[j], 0.0});
        Plaintext ptxt_w = cc->MakeCKKSPackedPlaintext(w_vec, 1, 0, nullptr, numSlots);

        auto ct_wx_j = cc->EvalMult(ct_feat[j], ptxt_w);
        cc->RescaleInPlace(ct_wx_j);

        if (first) { ct_dot = ct_wx_j; first = false; }
        else       { ct_dot = cc->EvalAdd(ct_dot, ct_wx_j); }
    }
    // ct_dot[i] = sum_j(w[j] * x[i][j]) = w^T x[i]  tum ornekler icin

    auto t_mult_end = Clock::now();
    double dt_mult = duration_cast<Ms>(t_mult_end - t_mult_start).count();
    std::cout << "[2] SIMD EvalMult+Add      : " << std::fixed << std::setprecision(2)
              << dt_mult << " ms  (tum ornekler paralel)" << std::endl;

    // ── ADIM 3: Bias — tum slot'lara ayni deger ───────────────────────────
    // bias skaler olarak tum slot'lara eklenir
    auto t_bias_start = Clock::now();
    auto ct_logit = cc->EvalAdd(ct_dot, bias);
    auto t_bias_end = Clock::now();
    double dt_bias = duration_cast<Ms>(t_bias_end - t_bias_start).count();
    std::cout << "[3] Bias ekleme (SIMD)     : " << std::fixed << std::setprecision(2)
              << dt_bias << " ms  (Level: " << ct_logit->GetLevel() << ")" << std::endl;

    // ── ADIM 4: Level hizalama ─────────────────────────────────────────────
    auto t_align_start = Clock::now();
    while (ct_logit->GetLevel() < targetLevel)
        cc->LevelReduceInPlace(ct_logit, nullptr);
    auto t_align_end = Clock::now();
    double dt_align = duration_cast<Ms>(t_align_end - t_align_start).count();
    std::cout << "[4] Level hizalama         : " << std::fixed << std::setprecision(2)
              << dt_align << " ms  (Level: " << ct_logit->GetLevel() << "/" << targetLevel << ")" << std::endl;

    // ── ADIM 5: TEK Bootstrap — 30 ornek paralel ───────────────────────────
    // EvalFuncBootstrap ct_logit uzerinde calisir
    // Her slot[i] bagimsiz olarak f(logit_i) hesaplanir — SIMD
    auto t_bs_start = Clock::now();
    auto ct_result = cc->EvalFuncBootstrap(ct_logit, f_lut, p, 1);
    auto t_bs_end = Clock::now();
    double dt_bs = duration_cast<Ms>(t_bs_end - t_bs_start).count();
    std::cout << "[5] EvalFuncBootstrap(SIMD): " << std::fixed << std::setprecision(2)
              << dt_bs << " ms  (1 kez, " << n_samples << " ornek paralel)" << std::endl;

    // ── ADIM 6: Tek Decrypt — tum sonuclar ────────────────────────────────
    auto t_dec_start = Clock::now();
    Plaintext pt_result;
    cc->Decrypt(keyPair.secretKey, ct_result, &pt_result);
    pt_result->SetLength(n_samples);
    auto t_dec_end = Clock::now();
    double dt_dec = duration_cast<Ms>(t_dec_end - t_dec_start).count();
    std::cout << "[6] Sifre cozme (tek)      : " << std::fixed << std::setprecision(2)
              << dt_dec << " ms" << std::endl;

    // ── Sonuclari ayikla ──────────────────────────────────────────────────
    auto fhe_values = pt_result->GetCKKSPackedValue();

    std::cout << "\n========================================" << std::endl;
    std::cout << "  Sonuclar  (slot[i] = ornek i)" << std::endl;
    std::cout << "========================================" << std::endl;

    int correct = 0;
    double total_infer = dt_enc + dt_mult + dt_bias + dt_align + dt_bs + dt_dec;

    for (int i = 0; i < n_samples; i++) {
        double fhe_val = fhe_values[i].real();
        int fhe_pred   = (fhe_val > 0.5) ? 1 : 0;
        bool match     = (fhe_pred == pred_pt_all[i]);
        if (match) correct++;

        std::cout << "  Ornek " << (i+1) << ": "
                  << "logit=" << std::fixed << std::setprecision(4) << (bias + [&](){
                        double s = 0;
                        for (int j = 0; j < n_features; j++) s += weights[j]*all_features[i][j];
                        return s;
                     }())
                  << "  FHE=" << std::fixed << std::setprecision(6) << fhe_val
                  << "  pred=" << fhe_pred
                  << "  gercek=" << true_labels[i]
                  << (match ? "  [OK]" : "  [HATA]") << std::endl;
    }

    std::cout << "\n========================================" << std::endl;
    std::cout << "  HESAPLAMA PROFILI  (SIMD, " << n_samples << " ornek)" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << std::fixed << std::setprecision(2);
    std::cout << "  Anahtar kurulum (toplam) : " << t_setup   << " ms" << std::endl;
    std::cout << "    -- Key uretimi         : " << t_keygen  << " ms" << std::endl;
    std::cout << "    -- Bootstrap key       : " << t_bskey   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Sifreli cikarim (toplam) : " << total_infer << " ms" << std::endl;
    std::cout << "    -- SIMD Sifreleme      : " << dt_enc   << " ms  (" << n_features << " ct)" << std::endl;
    std::cout << "    -- SIMD EvalMult+Add   : " << dt_mult  << " ms" << std::endl;
    std::cout << "    -- Bias ekleme         : " << dt_bias  << " ms" << std::endl;
    std::cout << "    -- Level hizalama      : " << dt_align << " ms" << std::endl;
    std::cout << "    -- EvalFuncBootstrap   : " << dt_bs    << " ms  (1 kez, " << n_samples << " paralel)" << std::endl;
    std::cout << "    -- Sifre cozme         : " << dt_dec   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Ornek basi amortize      : " << (total_infer / n_samples) << " ms" << std::endl;
    std::cout << "  Bootstrap orani          : " << (100.0 * dt_bs / total_infer) << " %" << std::endl;
    std::cout << "  ================================" << std::endl;
    std::cout << "  SONUC: " << correct << "/" << n_samples
              << "  Dogruluk: " << (100.0 * correct / n_samples) << " %" << std::endl;
    std::cout << "========================================\n" << std::endl;
}

int main() {
    IrisFunctionalBootstrapping();
    return 0;
}

--- End of file content ---
CMake Deprecation Warning at CMakeLists.txt:11 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Building in Release mode
CMake Warning (dev) at CMakeLists.txt:72 (option):
  Policy CMP0077 is not set: option() honors normal variables.  Run "cmake
  --help-policy CMP0077" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

  For compatibility with older versions of CMake, option is clearing the
  normal variable 'BUILD_UNITTESTS'.
This warning is for project developers.  Use -Wno-dev to suppress it.

CMake Warning (dev) at CMakeLists.txt:73 (option):
  Policy CMP0077 is not set: option() honors normal variables.  Run "cmake
  --help-policy CMP0077" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

  For compatibility with older versions of CMake, option is clearing the
  normal variable 'BUILD_EXAMPLES'.
This warning is for project developers.  Use -Wno-dev to suppress it.

CMake Warning (dev) at CMakeLists.txt:74 (option):
  Policy CMP0077 is not set: option() honors normal variables.  Run "cmake
  --help-policy CMP0077" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

  For compatibility with older versions of CMake, option is clearing the
  normal variable 'BUILD_BENCHMARKS'.
This warning is for project developers.  Use -Wno-dev to suppress it.

CMake Warning (dev) at CMakeLists.txt:78 (option):
  Policy CMP0077 is not set: option() honors normal variables.  Run "cmake
  --help-policy CMP0077" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

  For compatibility with older versions of CMake, option is clearing the
  normal variable 'BUILD_EXTRAS'.
This warning is for project developers.  Use -Wno-dev to suppress it.

-- BUILD_UNITTESTS:  ON
-- BUILD_EXAMPLES:   ON
-- BUILD_BENCHMARKS: ON
-- BUILD_EXTRAS:     OFF
-- BUILD_STATIC:     OFF
-- BUILD_SHARED:     ON
-- GIT_SUBMOD_AUTO:  ON
-- WITH_BE2:         OFF
-- WITH_BE4:         OFF
-- WITH_NTL:         OFF
-- WITH_TCM:         OFF
-- WITH_OPENMP:      ON
-- NATIVE_SIZE:      64
-- CKKS_M_FACTOR:    1
-- WITH_NATIVEOPT:   OFF
-- WITH_COVTEST:     OFF
-- WITH_NOISE_DEBUG: OFF
-- USE_MACPORTS:     OFF
-- BUILTIN_INFO_AVAILABLE is defined
***** INSTALL IS AT /usr/local; to change, run cmake with -DCMAKE_INSTALL_PREFIX=/your/path
-- Architecture is x86_64
-- Looking for sys/types.h
-- Looking for sys/types.h - found
-- Looking for stdint.h
-- Looking for stdint.h - found
-- Looking for stddef.h
-- Looking for stddef.h - found
-- Check size of __int128
-- Check size of __int128 - done
-- Check size of uint64_t
-- Check size of uint64_t - done
-- NATIVEINT is set to 64
-- MATHBACKEND is set to 4
-- MATHBACKEND set to 4. Setting WITH_BE4 to ON
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP: TRUE (found version "4.5")
-- Found Git: /usr/bin/git (found version "2.34.1")
-- Submodule update
Synchronizing submodule url for 'third-party/cereal'
Synchronizing submodule url for 'third-party/google-benchmark'
Synchronizing submodule url for 'third-party/google-test'
Synchronizing submodule url for 'third-party/gperftools'
CMake Deprecation Warning at third-party/google-benchmark/CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- Failed to find LLVM FileCheck
-- git version: v1.5.5-14-ge451e50e normalized to 1.5.5.14
-- Version: 1.5.5.14
-- Performing Test HAVE_CXX_FLAG_STD_CXX11
-- Performing Test HAVE_CXX_FLAG_STD_CXX11 - Success
-- Performing Test HAVE_CXX_FLAG_WALL
-- Performing Test HAVE_CXX_FLAG_WALL - Success
-- Performing Test HAVE_CXX_FLAG_WEXTRA
-- Performing Test HAVE_CXX_FLAG_WEXTRA - Success
-- Performing Test HAVE_CXX_FLAG_WSHADOW
-- Performing Test HAVE_CXX_FLAG_WSHADOW - Success
-- Performing Test HAVE_CXX_FLAG_WERROR
-- Performing Test HAVE_CXX_FLAG_WERROR - Success
-- Performing Test HAVE_CXX_FLAG_WSUGGEST_OVERRIDE
-- Performing Test HAVE_CXX_FLAG_WSUGGEST_OVERRIDE - Success
-- Performing Test HAVE_CXX_FLAG_PEDANTIC
-- Performing Test HAVE_CXX_FLAG_PEDANTIC - Success
-- Performing Test HAVE_CXX_FLAG_PEDANTIC_ERRORS
-- Performing Test HAVE_CXX_FLAG_PEDANTIC_ERRORS - Success
-- Performing Test HAVE_CXX_FLAG_WSHORTEN_64_TO_32
-- Performing Test HAVE_CXX_FLAG_WSHORTEN_64_TO_32 - Failed
-- Performing Test HAVE_CXX_FLAG_FSTRICT_ALIASING
-- Performing Test HAVE_CXX_FLAG_FSTRICT_ALIASING - Success
-- Performing Test HAVE_CXX_FLAG_WNO_DEPRECATED_DECLARATIONS
-- Performing Test HAVE_CXX_FLAG_WNO_DEPRECATED_DECLARATIONS - Success
-- Performing Test HAVE_CXX_FLAG_WNO_DEPRECATED
-- Performing Test HAVE_CXX_FLAG_WNO_DEPRECATED - Success
-- Performing Test HAVE_CXX_FLAG_WSTRICT_ALIASING
-- Performing Test HAVE_CXX_FLAG_WSTRICT_ALIASING - Success
-- Performing Test HAVE_CXX_FLAG_WD654
-- Performing Test HAVE_CXX_FLAG_WD654 - Failed
-- Performing Test HAVE_CXX_FLAG_WTHREAD_SAFETY
-- Performing Test HAVE_CXX_FLAG_WTHREAD_SAFETY - Failed
-- Performing Test HAVE_CXX_FLAG_COVERAGE
-- Performing Test HAVE_CXX_FLAG_COVERAGE - Success
-- Performing Test HAVE_STD_REGEX
-- Performing Test HAVE_STD_REGEX
-- Performing Test HAVE_STD_REGEX -- success
-- Performing Test HAVE_GNU_POSIX_REGEX
-- Performing Test HAVE_GNU_POSIX_REGEX
-- Performing Test HAVE_GNU_POSIX_REGEX -- failed to compile
-- Performing Test HAVE_POSIX_REGEX
-- Performing Test HAVE_POSIX_REGEX
-- Performing Test HAVE_POSIX_REGEX -- success
-- Performing Test HAVE_STEADY_CLOCK
-- Performing Test HAVE_STEADY_CLOCK
-- Performing Test HAVE_STEADY_CLOCK -- success
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Configuring done (6.6s)
-- Generating done (0.2s)
-- Build files have been written to: /content/sec-ckks/build
-- Copied demoData files
[  0%] Built target third-party
[  0%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/lattice/constants-lattice-impl.cpp.o
[  3%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/lattice/lattice.cpp.o
[  3%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/lattice/stdlatticeparms.cpp.o
[  3%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/lattice/trapdoor-dcrtpoly.cpp.o
[  3%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/lattice/trapdoor-poly.cpp.o
[  6%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/chebyshev.cpp.o
[  6%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/dftransform.cpp.o
[  9%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/distributiongenerator.cpp.o
[  9%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/hal/bigintdyn/be4-math-impl.cpp.o
[  9%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/hal/bigintdyn/mubintvecdyn.cpp.o
[  9%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/discretegaussiangeneratorgeneric.cpp.o
[  9%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/hal/bigintdyn/ubintdyn.cpp.o
[ 12%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/hal/bigintfxd/be2-math-impl.cpp.o
[ 12%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/hal/bigintfxd/mubintvecfxd.cpp.o
[ 12%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/hal/bigintfxd/ubintfxd.cpp.o
[ 15%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/hal/bigintntl/be6-math-impl.cpp.o
[ 15%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/hal/bigintntl/mubintvecntl.cpp.o
[ 15%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/hal/bigintntl/ubintntl.cpp.o
[ 15%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/hal/intnat/benative-math-impl.cpp.o
[ 18%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/hal/intnat/mubintvecnat.cpp.o
[ 18%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/hermite.cpp.o
[ 18%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/matrix.cpp.o
[ 21%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/math/nbtheory2.cpp.o
[ 21%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/utils/blockAllocator/blockAllocator.cpp.o
[ 21%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/utils/blockAllocator/xallocator.cpp.o
[ 21%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/utils/debug.cpp.o
[ 25%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/utils/demangle.cpp.o
[ 25%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/utils/get-call-stack.cpp.o
[ 25%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/utils/hashutil.cpp.o
[ 28%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/utils/inttypes.cpp.o
[ 28%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/utils/memory.cpp.o
[ 28%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/utils/openfhebase64.cpp.o
[ 28%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/utils/parallel.cpp.o
[ 31%] Building C object src/core/CMakeFiles/coreobj.dir/lib/utils/prng/blake2b-ref.c.o
[ 31%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/utils/prng/blake2engine.cpp.o
[ 31%] Building C object src/core/CMakeFiles/coreobj.dir/lib/utils/prng/blake2xb-ref.c.o
[ 34%] Building CXX object src/core/CMakeFiles/coreobj.dir/lib/utils/utilities.cpp.o
[ 34%] Built target coreobj
[ 37%] Linking CXX shared library ../../lib/libOPENFHEcore.so
[ 37%] Built target OPENFHEcore
[ 37%] Building CXX object src/binfhe/CMakeFiles/binfheobj.dir/lib/binfhe-base-scheme.cpp.o
[ 37%] Building CXX object src/binfhe/CMakeFiles/binfheobj.dir/lib/binfhe-constants-impl.cpp.o
[ 37%] Building CXX object src/binfhe/CMakeFiles/binfheobj.dir/lib/binfhecontext.cpp.o
[ 40%] Building CXX object src/binfhe/CMakeFiles/binfheobj.dir/lib/lwe-pke.cpp.o
[ 40%] Building CXX object src/binfhe/CMakeFiles/binfheobj.dir/lib/rgsw-acc-cggi.cpp.o
[ 40%] Building CXX object src/binfhe/CMakeFiles/binfheobj.dir/lib/rgsw-acc-dm.cpp.o
[ 43%] Building CXX object src/binfhe/CMakeFiles/binfheobj.dir/lib/rgsw-acc-lmkcdey.cpp.o
[ 43%] Building CXX object src/binfhe/CMakeFiles/binfheobj.dir/lib/rgsw-acc.cpp.o
[ 43%] Building CXX object src/binfhe/CMakeFiles/binfheobj.dir/lib/rgsw-cryptoparameters.cpp.o
[ 43%] Built target binfheobj
[ 43%] Linking CXX shared library ../../lib/libOPENFHEbinfhe.so
[ 43%] Built target OPENFHEbinfhe
[ 46%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/ciphertext-impl.cpp.o
[ 46%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/constants-impl.cpp.o
[ 50%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/cryptocontextfactory.cpp.o
[ 50%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/cryptocontext.cpp.o
[ 50%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/cryptoobject-impl.cpp.o
[ 50%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/encoding/ckkspackedencoding.cpp.o
[ 50%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/encoding/coefpackedencoding.cpp.o
[ 53%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/encoding/encodingparams.cpp.o
[ 53%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/encoding/packedencoding.cpp.o
[ 53%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/encoding/stringencoding.cpp.o
[ 56%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/globals-impl.cpp.o
[ 56%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/key/evalkey.cpp.o
[ 56%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/key/evalkeyrelin.cpp.o
[ 56%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/key/privatekey.cpp.o
[ 59%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/key/publickey.cpp.o
[ 59%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/keyswitch/keyswitch-base.cpp.o
[ 59%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/keyswitch/keyswitch-bv.cpp.o
[ 62%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/keyswitch/keyswitch-hybrid.cpp.o
[ 62%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/bfvrns/bfvrns-cryptoparameters.cpp.o
[ 62%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/bfvrns/bfvrns-leveledshe.cpp.o
[ 62%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/bfvrns/bfvrns-multiparty.cpp.o
[ 65%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/bfvrns/bfvrns-parametergeneration.cpp.o
[ 65%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/bfvrns/bfvrns-pke.cpp.o
[ 65%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/bfvrns/bfvrns-scheme.cpp.o
[ 65%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/bgvrns/bgvrns-cryptoparameters.cpp.o
[ 68%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/bgvrns/bgvrns-leveledshe.cpp.o
[ 68%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/bgvrns/bgvrns-multiparty.cpp.o
[ 68%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/bgvrns/bgvrns-parametergeneration.cpp.o
[ 71%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/bgvrns/bgvrns-pke.cpp.o
[ 71%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/bgvrns/bgvrns-scheme.cpp.o
[ 71%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/ckksrns/ckksrns-advancedshe.cpp.o
[ 71%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/ckksrns/ckksrns-cryptoparameters.cpp.o
[ 75%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/ckksrns/ckksrns-fhe.cpp.o
[ 75%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/ckksrns/ckksrns-leveledshe.cpp.o
[ 75%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/ckksrns/ckksrns-multiparty.cpp.o
[ 78%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/ckksrns/ckksrns-parametergeneration.cpp.o
[ 78%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/ckksrns/ckksrns-pke.cpp.o
[ 78%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/ckksrns/ckksrns-scheme.cpp.o
[ 78%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/ckksrns/ckksrns-schemeswitching.cpp.o
[ 81%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/ckksrns/ckksrns-utils.cpp.o
[ 81%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/ckksrns/schemeswitching-data-serializer.cpp.o
[ 81%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/gen-cryptocontext-params-impl.cpp.o
[ 84%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/gen-cryptocontext-params-validation.cpp.o
[ 84%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/scheme-id-impl.cpp.o
[ 84%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/scheme/scheme-swch-params.cpp.o
[ 84%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemebase/base-advancedshe.cpp.o
[ 87%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemebase/base-cryptoparameters.cpp.o
[ 87%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemebase/base-fhe.cpp.o
[ 87%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemebase/base-leveledshe.cpp.o
[ 90%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemebase/base-multiparty.cpp.o
[ 90%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemebase/base-parametergeneration.cpp.o
[ 90%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemebase/base-pke.cpp.o
[ 90%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemebase/base-pre.cpp.o
[ 93%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemebase/base-scheme.cpp.o
[ 93%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemebase/rlwe-cryptoparameters-impl.cpp.o
[ 93%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemerns/rns-cryptoparameters.cpp.o
[ 96%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemerns/rns-leveledshe.cpp.o
[ 96%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemerns/rns-multiparty.cpp.o
[ 96%] Building CXX object src/pke/CMakeFiles/pkeobj.dir/lib/schemerns/rns-pke.cpp.o
[ 96%] Built target pkeobj
[ 96%] Linking CXX shared library ../../lib/libOPENFHEpke.so
[ 96%] Built target OPENFHEpke
[100%] Building CXX object src/pke/CMakeFiles/iris_functionallite.dir/examples/iris_functionallite.cpp.o
[100%] Linking CXX executable ../../bin/examples/pke/iris_functionallite
[100%] Built target iris_functionallite

========================================
  SIMD CKKS — Anahtar Kurulum
========================================
  p (modul)           : 32  (guvenli aralik: (-16, +16))
  bits                : 5
  depth               : 23
  Ring boyutu (n)     : 65536
  Slot sayisi         : 32768
  Carpim derinligi    : 23
  Key uretimi         : 1114.97 ms
  Bootstrap key       : 21981.39 ms
  Toplam kurulum      : 36793.95 ms

  Plaintext logitler (dogrulama):
    Ornek 1: logit=-4.3292  [OK]  pred=0
    Ornek 2: logit=3.5396  [OK]  pred=1
    Ornek 3: logit=-11.1618  [OK]  pred=0
    Ornek 4: logit=-4.3519  [OK]  pred=0
    Ornek 5: logit=-5.6923  [OK]  pred=0
    Ornek 6: logit=2.8185  [OK]  pred=1
    Ornek 7: logit=-2.5971  [OK]  pred=0
    Ornek 8: logit=-7.1113  [OK]  pred=0
    Ornek 9: logit=-6.4881  [OK]  pred=0
    Ornek 10: logit=-3.4672  [OK]  pred=0
    Ornek 11: logit=-5.7414  [OK]  pred=0
    Ornek 12: logit=3.1808  [OK]  pred=1
    Ornek 13: logit=3.5597  [OK]  pred=1
    Ornek 14: logit=3.2321  [OK]  pred=1
    Ornek 15: logit=4.4704  [OK]  pred=1
    Ornek 16: logit=-4.0351  [OK]  pred=0
    Ornek 17: logit=-7.3521  [OK]  pred=0
    Ornek 18: logit=-3.5580  [OK]  pred=0
    Ornek 19: logit=-3.8473  [OK]  pred=0
    Ornek 20: logit=-7.5780  [OK]  pred=0
    Ornek 21: logit=3.4514  [OK]  pred=1
    Ornek 22: logit=-5.1929  [OK]  pred=0
    Ornek 23: logit=3.2168  [OK]  pred=1
    Ornek 24: logit=-7.3759  [OK]  pred=0
    Ornek 25: logit=-7.0870  [OK]  pred=0
    Ornek 26: logit=-7.2297  [OK]  pred=0
    Ornek 27: logit=-8.1397  [OK]  pred=0
    Ornek 28: logit=-7.4799  [OK]  pred=0
    Ornek 29: logit=2.7764  [OK]  pred=1
    Ornek 30: logit=3.0581  [OK]  pred=1

========================================
  SIMD Batch Sifreli Cikarim
  Strateji: ct_feat[j][slot_i] = ornek_i ozellik_j
  30 ornek PARALEL, 1 bootstrap
========================================

[1] SIMD Batch Sifreleme   : 946.89 ms  (4 ct, 30 ornek paralel)
[2] SIMD EvalMult+Add      : 378.76 ms  (tum ornekler paralel)
[3] Bias ekleme (SIMD)     : 5.28 ms  (Level: 1)
[4] Level hizalama         : 0.12 ms  (Level: 21/21)
[5] EvalFuncBootstrap(SIMD): 28928.02 ms  (1 kez, 30 ornek paralel)
[6] Sifre cozme (tek)      : 36.51 ms

========================================
  Sonuclar  (slot[i] = ornek i)
========================================
  Ornek 1: logit=-4.3292  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 2: logit=3.5396  FHE=1.000000  pred=1  gercek=1  [OK]
  Ornek 3: logit=-11.1618  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 4: logit=-4.3519  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 5: logit=-5.6923  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 6: logit=2.8185  FHE=1.000000  pred=1  gercek=1  [OK]
  Ornek 7: logit=-2.5971  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 8: logit=-7.1113  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 9: logit=-6.4881  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 10: logit=-3.4672  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 11: logit=-5.7414  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 12: logit=3.1808  FHE=1.000000  pred=1  gercek=1  [OK]
  Ornek 13: logit=3.5597  FHE=1.000000  pred=1  gercek=1  [OK]
  Ornek 14: logit=3.2321  FHE=1.000000  pred=1  gercek=1  [OK]
  Ornek 15: logit=4.4704  FHE=1.000000  pred=1  gercek=1  [OK]
  Ornek 16: logit=-4.0351  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 17: logit=-7.3521  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 18: logit=-3.5580  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 19: logit=-3.8473  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 20: logit=-7.5780  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 21: logit=3.4514  FHE=1.000000  pred=1  gercek=1  [OK]
  Ornek 22: logit=-5.1929  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 23: logit=3.2168  FHE=1.000000  pred=1  gercek=1  [OK]
  Ornek 24: logit=-7.3759  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 25: logit=-7.0870  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 26: logit=-7.2297  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 27: logit=-8.1397  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 28: logit=-7.4799  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 29: logit=2.7764  FHE=1.000000  pred=1  gercek=1  [OK]
  Ornek 30: logit=3.0581  FHE=1.000000  pred=1  gercek=1  [OK]

========================================
  HESAPLAMA PROFILI  (SIMD, 30 ornek)
========================================
  Anahtar kurulum (toplam) : 36793.95 ms
    -- Key uretimi         : 1114.97 ms
    -- Bootstrap key       : 21981.39 ms
  --------------------------------
  Sifreli cikarim (toplam) : 30295.59 ms
    -- SIMD Sifreleme      : 946.89 ms  (4 ct)
    -- SIMD EvalMult+Add   : 378.76 ms
    -- Bias ekleme         : 5.28 ms
    -- Level hizalama      : 0.12 ms
    -- EvalFuncBootstrap   : 28928.02 ms  (1 kez, 30 paralel)
    -- Sifre cozme         : 36.51 ms
  --------------------------------
  Ornek basi amortize      : 1009.85 ms
  Bootstrap orani          : 95.49 %
  ================================
  SONUC: 30/30  Dogruluk: 100.00 %
========================================



In [ ]:
/content/sec-ckks
[ 96%] Built target pkeobj
[ 96%] Built target OPENFHEpke
[ 96%] Building CXX object src/pke/CMakeFiles/iris_functional.dir/examples/iris_functional.cpp.o
[100%] Linking CXX executable ../../bin/examples/pke/iris_functional
[100%] Built target iris_functional

========================================
  SIMD CKKS — Anahtar Kurulum
========================================
  p (modul)           : 32  (guvenli aralik: (-16, +16))
  bits                : 5
  depth               : 23
  Ring boyutu (n)     : 8192
  Slot sayisi         : 4096
  Carpim derinligi    : 23
  Key uretimi         : 136.94 ms
  Bootstrap key       : 1580.99 ms
  Toplam kurulum      : 2480.35 ms

  Plaintext logitler (dogrulama):
    Ornek 1: logit=-4.3283  [OK]  pred=0
    Ornek 2: logit=3.5417  [OK]  pred=1
    Ornek 3: logit=-11.1625  [OK]  pred=0
    Ornek 4: logit=-4.3528  [OK]  pred=0
    Ornek 5: logit=-5.6910  [OK]  pred=0

========================================
  SIMD Batch Sifreli Cikarim
  Strateji: ct_feat[j][slot_i] = ornek_i ozellik_j
  5 ornek PARALEL, 1 bootstrap
========================================

[1] SIMD Batch Sifreleme   : 110.99 ms  (4 ct, 5 ornek paralel)
[2] SIMD EvalMult+Add      : 43.10 ms  (tum ornekler paralel)
[3] Bias ekleme (SIMD)     : 0.53 ms  (Level: 1)
[4] Level hizalama         : 0.09 ms  (Level: 21/21)
[5] EvalFuncBootstrap(SIMD): 2462.26 ms  (1 kez, 5 ornek paralel)
[6] Sifre cozme (tek)      : 4.83 ms

========================================
  Sonuclar  (slot[i] = ornek i)
========================================
  Ornek 1: logit=-4.3283  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 2: logit=3.5417  FHE=1.000000  pred=1  gercek=1  [OK]
  Ornek 3: logit=-11.1625  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 4: logit=-4.3528  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 5: logit=-5.6910  FHE=0.000000  pred=0  gercek=0  [OK]

========================================
  HESAPLAMA PROFILI  (SIMD, 5 ornek)
========================================
  Anahtar kurulum (toplam) : 2480.35 ms
    -- Key uretimi         : 136.94 ms
    -- Bootstrap key       : 1580.99 ms
  --------------------------------
  Sifreli cikarim (toplam) : 2621.79 ms
    -- SIMD Sifreleme      : 110.99 ms  (4 ct)
    -- SIMD EvalMult+Add   : 43.10 ms
    -- Bias ekleme         : 0.53 ms
    -- Level hizalama      : 0.09 ms
    -- EvalFuncBootstrap   : 2462.26 ms  (1 kez, 5 paralel)
    -- Sifre cozme         : 4.83 ms
  --------------------------------
  Ornek basi amortize      : 524.36 ms
  Bootstrap orani          : 93.92 %
  ================================
  SONUC: 5/5  Dogruluk: 100.00 %
========================================